## Problem Setting

**CDSP** (Clinic Drug Service Problem): scheduled drug delivery to medical clinics, each with a narrow time window [rt, due].

**VRP-D extension**: every truck leg can launch a drone via a k=depot sortie ⟨i, j, 0⟩ — drone flies i→j→depot while the truck returns directly. This serves an additional clinic per leg with no extra vehicle needed.

Key parameters: `drone_speed = 1.43×`, `endurance = 30`, `SL = SR = 1`.  
Objective: minimise Σ(C[j] − rt[j])  where C[j] = `drone_at_j` for drone customers, `return_time` for truck customers.


In [16]:
import numpy as np
import math
import random
import time
import csv
import os
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from pathlib import Path
from scipy.spatial.distance import cdist


## 1. Dataset Preparation

### 1.1 Instance loading

In [17]:
# ── Data loading (robust parser) ──────────────────────────────────────────────

def load_instance(filepath):
    """
    Parse a CDSP benchmark instance file into a structured dictionary.

    File format
    -----------
    Line 0 : <char> <num_nodes> <max_vehicles>
        - num_nodes   : total node count including the depot (node 0)
        - max_vehicles: fleet size upper bound
    Lines 1+: <x> <y> <ready_time> <due_date>
        - Node 0 is the depot; nodes 1..num_nodes-1 are clinics (requests).
        - ready_time : earliest moment the vehicle may serve this node
        - due_date   : latest moment the vehicle must arrive at this node
        - max_tau is fixed at 480 (minutes) — the maximum shift duration per leg

    Returns
    -------
    data_dict with keys: coords, ready_times, due_dates,
                         max_vehicles, max_tau, num_nodes
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()

    header       = lines[0].strip().split()
    num_nodes    = int(header[1])
    max_vehicles = int(header[2])
    max_tau      = 480          # fixed shift-duration limit (minutes)

    node_data = np.loadtxt(lines[1:])

    return {
        'coords'      : node_data[:, 0:2],
        'ready_times' : node_data[:, 2],
        'due_dates'   : node_data[:, 3],
        'max_vehicles': max_vehicles,
        'max_tau'     : max_tau,
        'num_nodes'   : num_nodes,
    }


def preprocess_instance(data_dict):
    """Prune arcs where ready_time[i] + dist(i,j) > due_date[j] (set to inf)."""
    coords      = data_dict['coords']
    ready_times = data_dict['ready_times']
    due_dates   = data_dict['due_dates']

    dist_matrix  = cdist(coords, coords, metric='euclidean')
    arrival_at_j = ready_times[:, np.newaxis] + dist_matrix
    invalid_mask = arrival_at_j > due_dates[np.newaxis, :]
    np.fill_diagonal(invalid_mask, True)
    processed = dist_matrix.copy()
    processed[invalid_mask] = np.inf
    data_dict['dist_matrix'] = processed
    return data_dict


DEFAULT_DRONE_PARAMS = {
    'drone_speed_factor': 1.43,
    'endurance'         : 30.0,
    'SL'                : 0.0,
    'SR'                : 0.0,
}


def add_drone_params(data_dict, drone_params=None):
    """Add drone distance matrix (dist / speed_factor) and params to data_dict."""
    params   = {**DEFAULT_DRONE_PARAMS, **(drone_params or {})}
    coords   = data_dict['coords']
    dist_raw = cdist(coords, coords, metric='euclidean')
    np.fill_diagonal(dist_raw, 0.0)
    data_dict['dist_raw']     = dist_raw
    data_dict['dist_drone']   = dist_raw / params['drone_speed_factor']
    data_dict['drone_params'] = params
    return data_dict


def load_and_prepare(filepath, drone_params=None):
    """Convenience wrapper: load → preprocess → add_drone_params."""
    data = load_instance(filepath)
    data = preprocess_instance(data)
    data = add_drone_params(data, drone_params)
    return data


### 1.2 Sortie set

In [18]:
def build_sortie_set(data_dict) -> Tuple[List[Tuple], Dict[int, List]]:
    """Enumerate battery-feasible sorties ⟨i, j, k⟩.
    k=0 (depot-return) is the only viable type for single-customer CDSP legs.
    Battery check: SL + dd[i,j] + dd[j,k] + SR ≤ endurance.
    Returns (sortie_set, sortie_idx) where sortie_idx[j] lists all sorties to j.
    """
    N   = data_dict['num_nodes']
    dd  = data_dict['dist_drone']
    p   = data_dict['drone_params']
    SL, SR, end = p['SL'], p['SR'], p['endurance']

    sortie_set = []
    sortie_idx: Dict[int, List] = {}

    for j in range(1, N):                   # j is always a customer
        for i in range(1, N):
            if i == j: continue
            for k in range(0, N):            # k=0 = depot
                if k == j or k == i: continue
                if SL + dd[i, j] + dd[j, k] + SR <= end + 1e-9:
                    s = (i, j, k)
                    sortie_set.append(s)
                    sortie_idx.setdefault(j, []).append(s)

    return sortie_set, sortie_idx


### 1.3 Solution representation

In [19]:
@dataclass
class RouteGroup:
    """One vehicle\'s assignment: legs = list of customer groups between depot returns."""
    legs: List[List[int]] = field(default_factory=list)

    def all_jobs(self) -> List[int]:
        return [j for leg in self.legs for j in leg]

    def n_replenishments(self) -> int:
        return max(0, len(self.legs) - 1)

    def is_empty(self) -> bool:
        return all(len(leg) == 0 for leg in self.legs)

    def copy(self) -> 'RouteGroup':
        return RouteGroup(legs=[list(leg) for leg in self.legs])


In [20]:
# ============================================================================
# VALIDATED TRUCK-ONLY BASELINE (copied from final truck notebook)
# Namespaced with truck_ to avoid collisions with the drone ALNS.
# ============================================================================


def truck_mip_build_and_solve(data_dict, time_limit=3600, solver_name='GUROBI',
                              threads=4, verbose=False, use_warm_start=False):
    """Strict Rocha Gurobi MIP used only for truck-only restricted intensification."""
    if solver_name != 'GUROBI':
        raise ValueError("v6 truck baseline uses the strict Gurobi formulation only.")
    return truck_mip_solve_gurobi(
        data_dict,
        time_limit=time_limit,
        threads=threads,
        verbose=verbose,
        use_warm_start=use_warm_start
    )

import gurobipy as gp
from gurobipy import GRB

def truck_greedy_nearest_neighbor(data_dict):
    """
    Build a feasible solution by repeatedly extending each vehicle's route with
    the nearest clinic that can still be reached within its time window AND
    allows the vehicle to return to depot within the shift limit (max_tau).

    Key feasibility checks per candidate j
    ---------------------------------------
    1. Arc i→j must be finite (pruned by preprocess_instance).
    2. Arrival at j <= due_date[j].
    3. arrival[j] + dist(j, depot) <= depart_time + max_tau
       (the vehicle's total shift must not exceed max_tau minutes from departure).

    Departure time tracking
    -----------------------
    `depart_time` is computed once per route from the first clinic served:
        depart_time = max(0, ready_time[first_j] - dist(depot, first_j))
    This is the earliest the vehicle can leave the depot to arrive exactly at
    ready_time[first_j], and is used to enforce the shift-duration constraint.

    Unvisited clinics
    -----------------
    If a clinic cannot be added to any vehicle, it is left in `unvisited` and
    penalised in the objective as:
        due_date[j] + dist(j, depot) - ready_time[j]
    (an upper-bound proxy for what its completion time would cost).
    """
    dist = data_dict['dist_matrix']
    rt   = data_dict['ready_times']
    dd   = data_dict['due_dates']
    K    = data_dict['max_vehicles']
    T    = data_dict['max_tau']
    N    = data_dict['num_nodes']
    depot = 0

    unvisited = set(range(1, N))
    routes, total_C = [], 0.0

    for _ in range(K):
        if not unvisited:
            break
        route, current, curr_time = [], depot, 0.0
        depart_time = None

        while unvisited:
            best_j, best_arr = None, np.inf
            for j in unvisited:
                if np.isinf(dist[current, j]):
                    continue
                arr = max(curr_time + dist[current, j], rt[j])
                # Use known departure time for shift check; estimate it for the first node
                dep = depart_time if depart_time is not None else max(0.0, rt[j] - dist[depot, j])
                if arr <= dd[j] and arr + dist[j, depot] <= dep + T and arr < best_arr:
                    best_j, best_arr = j, arr
            if best_j is None:
                break
            if depart_time is None:
                depart_time = max(0.0, best_arr - dist[depot, best_j])
            route.append(best_j)
            unvisited.discard(best_j)
            curr_time, current = best_arr, best_j

        if route:
            return_time = curr_time + dist[current, depot]
            # Corrected completion-time contribution: Σ(C[j] - r[j]) for this route
            # C[j] = return_time for every j in the same leg (all share the depot return)
            total_C += return_time * len(route) - sum(rt[j] for j in route)
            routes.append((route, return_time))

    # Penalty for clinics that no vehicle could serve
    if unvisited:
        penalty = sum(
            dd[j] + (dist[j, depot] if not np.isinf(dist[j, depot]) else 0.0) - rt[j]
            for j in unvisited
        )
        total_C += penalty

    return routes, total_C, list(unvisited)

def truck_mip_solve_gurobi(data_dict, time_limit=3600, threads=4,
                      verbose=True, use_warm_start=False):
    """
    Exact two-index replenishment-arc formulation of Rocha et al. (2025).

    This implementation mirrors constraints (1)-(14) of the anchor paper.
    In particular it uses:
      x_e      unified binary arc variable on original and replenishment arcs
      y[i,j]   specimen-tracking binary variable
      z[j]     absolute visit time at request j
      tau[j]   elapsed time since the vehicle's first departure from the depot
      C[j]     request completion time (arrival at the laboratory)

    The benchmark-reported objective is F' = sum_j (C[j] - r[j]).  This differs
    from the mathematical objective sum C[j] only by an instance-specific
    constant, so it has exactly the same minimizers and LP relaxation strength.

    The paper-specific preprocessing is applied locally here:
        r_j <- max(r_j, c_0j)
        d_j <- min(d_j, d_0 - c_j0)
    This is intentionally kept separate from preprocess_instance(), which is
    also used by the ALNS and performs additional arc pruning.

    Notes on warm starts
    --------------------
    The anchor paper reports standard Gurobi parameters and does not describe a
    MIP start.  To keep the replication clean, warm starts are disabled by
    default.  If use_warm_start=True is passed, it is currently ignored rather
    than injecting a partial/possibly non-equivalent start.
    """
    coords = np.asarray(data_dict['coords'], dtype=float)
    rt0    = np.asarray(data_dict['ready_times'], dtype=float)
    dd0    = np.asarray(data_dict['due_dates'], dtype=float)
    K      = int(data_dict['max_vehicles'])
    T      = float(data_dict['max_tau'])
    N      = int(data_dict['num_nodes'])

    depot    = 0
    requests = list(range(1, N))

    # Rocha's graph is complete on the original arcs before time-window bounds
    # are enforced.  Rebuild Euclidean distances from coordinates so that the
    # ALNS-specific inf arc pruning in data_dict['dist_matrix'] does not change
    # the exact formulation.
    raw_dist = cdist(coords, coords, metric='euclidean')

    # Paper preprocessing: tighten request windows with depot reachability.
    r = rt0.copy()
    d = dd0.copy()
    d0 = float(dd0[depot])
    for j in requests:
        r[j] = max(r[j], raw_dist[depot, j])
        d[j] = min(d[j], d0 - raw_dist[j, depot])
        if r[j] > d[j] + 1e-9:
            return {
                'status': 'Infeasible', 'objective': None, 'lower_bound': None,
                'mip_gap_pct': None, 'wall_time': 0.0,
                'proven_optimal': False, 'formulation': 'Rocha2025'
            }

    # Arc sets.  The type tag is required because AP and AR are parallel arcs
    # between the same ordered pair of requests.
    A0 = ([('O', depot, j) for j in requests] +
          [('O', j, depot) for j in requests])
    AP = [('O', i, j) for i in requests for j in requests if i != j]
    AR = [('R', i, j) for i in requests for j in requests if i != j]
    A  = A0 + AP + AR

    def src(e): return e[1]
    def tgt(e): return e[2]

    c = {}
    for e in A0 + AP:
        c[e] = float(raw_dist[src(e), tgt(e)])
    for e in AR:
        i, j = src(e), tgt(e)
        c[e] = float(raw_dist[i, depot] + raw_dist[depot, j])

    out_arcs = {j: [e for e in A if src(e) == j] for j in range(N)}
    in_arcs  = {j: [e for e in A if tgt(e) == j] for j in range(N)}

    # Big-M coefficients exactly as defined in Rocha et al. (2025).
    M_e = {
        e: max(0.0, float(d[src(e)] + c[e] - r[tgt(e)]))
        for e in list(AP) + list(AR)
    }
    M_i = {
        i: float(d[i] + raw_dist[i, depot])
        for i in requests
    }
    Mprime_e = {
        e: float(d[tgt(e)] - raw_dist[depot, tgt(e)])
        for e in list(AP) + list(AR)
    }

    if use_warm_start and verbose:
        print('  Note: use_warm_start=True ignored for Rocha2025 replication.')

    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 1 if verbose else 0)
    env.start()
    m = gp.Model(env=env)
    m.setParam('TimeLimit', time_limit)
    m.setParam('Threads', threads)

    # (12)-(14) variable domains.
    x   = m.addVars(A, vtype=GRB.BINARY, name='x')
    y   = m.addVars(requests, requests, vtype=GRB.BINARY, name='y')
    z   = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='z')
    tau = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='tau')
    C   = m.addVars(requests, lb=0.0, vtype=GRB.CONTINUOUS, name='C')

    # (1) Same minimizers as sum C_j, reported on Rocha's corrected F' scale.
    m.setObjective(gp.quicksum(C[j] - r[j] for j in requests), GRB.MINIMIZE)

    # (2) Depot flow balance and fleet limit.
    m.addConstr(
        gp.quicksum(x[e] for e in out_arcs[depot]) ==
        gp.quicksum(x[e] for e in in_arcs[depot]),
        name='c2_depot_balance')
    m.addConstr(
        gp.quicksum(x[e] for e in out_arcs[depot]) <= K,
        name='c2_fleet_limit')

    # (3) Every request has exactly one incoming and one outgoing arc in the
    # multigraph (original or replenishment).
    for j in requests:
        m.addConstr(gp.quicksum(x[e] for e in out_arcs[j]) == 1,
                    name=f'c3_out_{j}')
        m.addConstr(gp.quicksum(x[e] for e in in_arcs[j]) == 1,
                    name=f'c3_in_{j}')

    # (4) Visit-time propagation over original request-request arcs and
    # replenishment arcs.  Write every constraint as lhs <= 0 to avoid
    # NumPy/Gurobi comparison ambiguity.
    for e in list(AP) + list(AR):
        i = src(e)
        j = tgt(e)
        lhs = (
            z[i]
            + float(c[e])
            - z[j]
            - float(M_e[e]) * (1 - x[e])
        )
        m.addConstr(lhs <= 0, name=f'c4_{e[0]}_{i}_{j}')

    # (5) Tightened hard time windows.
    for j in requests:
        m.addConstr(z[j] >= float(r[j]), name=f'c5_r_{j}')
        m.addConstr(z[j] <= float(d[j]), name=f'c5_d_{j}')

    # (6) A request is on the vehicle immediately after collection.
    for j in requests:
        m.addConstr(y[j, j] == 1, name=f'c6_{j}')

    # (7) Propagate specimen j along ORIGINAL request-request arcs until the
    # vehicle next returns to the depot.  Replenishment arcs deliberately do
    # not propagate y because they contain a depot visit.
    for e in list(AP):
        i = src(e)
        k = tgt(e)
        for req in requests:
            if req == k:
                continue
            lhs = y[i, req] - y[k, req] - (1 - x[e])
            m.addConstr(lhs <= 0, name=f'c7_{i}_{k}_{req}')

    # (8) Completion time: if specimen req is still on the vehicle at i, its
    # delivery cannot occur before travelling i -> depot.
    for i in requests:
        return_to_depot = float(raw_dist[i, depot])
        Mi = float(M_i[i])
        for req in requests:
            lhs = (
                z[i]
                + return_to_depot
                - C[req]
                - Mi * (1 - y[i, req])
            )
            m.addConstr(lhs <= 0, name=f'c8_{i}_{req}')

    # (9) Relative visit time cannot be smaller than a direct depot -> j trip.
    # Put the Gurobi variable on the left-hand side; this avoids NumPy scalar
    # comparison creating an ambiguous TempConstr boolean.
    for j in requests:
        m.addConstr(
            tau[j] >= float(raw_dist[depot, j]),
            name=f'c9_{j}'
        )

    # (10) Propagate elapsed shift time over both AP and AR.  For a
    # replenishment arc, z[j]-z[i] includes the intermediate depot visit and
    # any waiting, so the shift clock does not reset between trips.
    for e in list(AP) + list(AR):
        i = src(e)
        j = tgt(e)
        lhs = (
            tau[i]
            + z[j]
            - z[i]
            - tau[j]
            - float(Mprime_e[e]) * (1 - x[e])
        )
        m.addConstr(lhs <= 0, name=f'c10_{e[0]}_{i}_{j}')

    # (11) Full vehicle shift from first departure to final depot return.
    for j in requests:
        m.addConstr(
            tau[j] + float(raw_dist[j, depot]) <= float(T),
            name=f'c11_{j}'
        )

    t0 = time.time()
    m.optimize()
    wall_time = time.time() - t0

    status_map = {
        GRB.OPTIMAL    : 'Optimal',
        GRB.INFEASIBLE : 'Infeasible',
        GRB.INF_OR_UNBD: 'Infeasible',
        GRB.TIME_LIMIT : 'Time limit',
        GRB.SUBOPTIMAL : 'Feasible',
    }
    status = status_map.get(m.Status, f'Status_{m.Status}')

    # Report Obj/LB on Rocha's original-release F' scale.
    reporting_shift = float(sum(r[j] - rt0[j] for j in requests))

    obj_val = (
        m.ObjVal + reporting_shift
        if m.SolCount > 0 else None
    )
    lower_bound = (
        None if m.Status == GRB.INFEASIBLE
        else m.ObjBound + reporting_shift
    )

    mip_gap = None
    if obj_val is not None and lower_bound is not None and abs(obj_val) > 1e-9:
        mip_gap = abs(obj_val - lower_bound) / abs(obj_val)

    truly_optimal = (m.Status == GRB.OPTIMAL and
                     (mip_gap is None or mip_gap < 1e-4))

    mip_legs = None
    mip_groups = None
    if m.SolCount > 0:
        mip_groups = truck_extract_groups_from_rocha_gurobi(x, requests, depot)
        mip_legs = [leg for grp in mip_groups for leg in grp]

    result = {
        'status'         : status,
        'objective'      : round(obj_val, 4) if obj_val is not None else None,
        'lower_bound'    : round(lower_bound, 4) if lower_bound is not None else None,
        'mip_gap_pct'    : round(mip_gap * 100, 2) if mip_gap is not None else None,
        'wall_time'      : round(wall_time, 2),
        'proven_optimal' : truly_optimal,
        'formulation'    : 'Rocha2025',
        'n_vars'         : m.NumVars,
        'n_constrs'      : m.NumConstrs,
    }
    if mip_legs is not None:
        result['legs'] = mip_legs
    if mip_groups is not None:
        result['route_groups'] = mip_groups

    m.dispose()
    env.dispose()
    return result

def truck_extract_groups_from_rocha_gurobi(x, requests, depot=0):
    """
    Reconstruct complete vehicle groups from Rocha's unified arc solution.

    Returns
    -------
    List[List[List[int]]]
        Outer list = vehicles.
        Middle list = depot-to-depot legs/trips of one vehicle.
        Inner list = request indices in one leg.

    A replenishment arc ('R', i, j) represents i -> depot -> j. It therefore
    closes the current leg but keeps the next leg on the SAME vehicle.
    """
    selected = {e for e in x.keys() if x[e].X > 0.5}
    starts = [j for j in requests if ('O', depot, j) in selected]

    route_groups = []
    globally_seen = set()

    for start in starts:
        cur = start
        vehicle_legs = []
        leg = []
        safety = 0

        while safety < 20 * (len(requests) + 1):
            safety += 1

            if cur in globally_seen:
                break

            globally_seen.add(cur)
            leg.append(cur)

            # Original request -> depot arc finishes this vehicle.
            if ('O', cur, depot) in selected:
                vehicle_legs.append(leg)
                leg = []
                break

            # Replenishment arc: finish this trip, continue same vehicle.
            next_repl = [
                j for j in requests
                if j != cur and ('R', cur, j) in selected
            ]

            # Original request -> request arc: remain on same trip.
            next_original = [
                j for j in requests
                if j != cur and ('O', cur, j) in selected
            ]

            if next_repl:
                vehicle_legs.append(leg)
                leg = []
                cur = next_repl[0]
                continue

            if next_original:
                cur = next_original[0]
                continue

            # Defensive fallback for an unexpected extraction state.
            if leg:
                vehicle_legs.append(leg)
            break

        if vehicle_legs:
            route_groups.append(vehicle_legs)

    return route_groups

def truck_extract_routes_from_rocha_gurobi(x, requests, depot):
    """Extract depot-to-depot legs from the unified Rocha arc variables."""
    selected = {e for e in x.keys() if x[e].X > 0.5}

    # Every route starts on an original depot->request arc.
    starts = [j for j in requests if ('O', depot, j) in selected]
    legs = []
    seen = set()

    for start in starts:
        cur = start
        leg = []
        safety = 0
        while cur != depot and safety < 10 * (len(requests) + 1):
            safety += 1
            # A request appears only once globally in a feasible solution.
            if cur in seen:
                break
            seen.add(cur)
            leg.append(cur)

            e_back = ('O', cur, depot)
            if e_back in selected:
                legs.append(leg)
                break

            next_original = [j for j in requests
                             if j != cur and ('O', cur, j) in selected]
            next_repl = [j for j in requests
                         if j != cur and ('R', cur, j) in selected]

            if next_repl:
                # The replenishment arc cur -> depot -> j closes the current
                # leg and immediately begins the next trip of the same vehicle.
                legs.append(leg)
                leg = []
                cur = next_repl[0]
            elif next_original:
                cur = next_original[0]
            else:
                # Defensive fallback for an unexpected extraction state.
                if leg:
                    legs.append(leg)
                break

    return legs

def truck_simulate_group(grp: RouteGroup, data_dict: dict, return_details=False):
    """
    Simulate ONE vehicle across all of its legs on one continuous clock.

    A RouteGroup represents one physical vehicle.  If its legs are
        [[1, 2], [3, 4]]
    the actual sequence is
        0 -> 1 -> 2 -> 0 -> 3 -> 4 -> 0
    and the second leg cannot start before the first leg has returned to depot.

    The first departure may be delayed so that the vehicle reaches the first
    clinic just in time.  This follows Rocha et al.'s shift definition: the
    shift starts when the vehicle leaves the depot for its first trip and ends
    when it returns after its last trip.

    Objective contribution is the corrected completion time
        sum_j (C[j] - r[j]),
    where C[j] is the depot-return time of the leg carrying request j.

    Returns
    -------
    If return_details=False:
        (cost, feasible)
    If return_details=True:
        (cost, feasible, details)
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0
    eps   = 1e-6

    vehicle_clock  = 0.0
    first_departure = None
    total = 0.0
    arrivals = {}
    completions = {}
    leg_returns = []
    leg_departures = []

    for leg in grp.legs:
        if not leg:
            continue

        first = leg[0]
        if np.isinf(dist[depot, first]):
            result = (np.inf, False)
            return (*result, None) if return_details else result

        # The vehicle is already at the depot after the preceding leg.
        # It may wait at the depot before starting the next trip.
        leg_depart = max(vehicle_clock, 0.0, rt[first] - dist[depot, first])
        if first_departure is None:
            first_departure = leg_depart

        cur_time, prev = leg_depart, depot
        for j in leg:
            if np.isinf(dist[prev, j]):
                result = (np.inf, False)
                return (*result, None) if return_details else result

            arr = max(cur_time + dist[prev, j], rt[j])
            if arr > dd[j] + eps:
                result = (np.inf, False)
                return (*result, None) if return_details else result

            arrivals[j] = arr
            cur_time, prev = arr, j

        if np.isinf(dist[prev, depot]):
            result = (np.inf, False)
            return (*result, None) if return_details else result

        return_time = cur_time + dist[prev, depot]

        # d0 is the latest return to the laboratory.  Since leg return times are
        # nondecreasing, checking every depot return is equivalent to checking
        # the final one and catches infeasibility as early as possible.
        if return_time > dd[depot] + eps:
            result = (np.inf, False)
            return (*result, None) if return_details else result

        # FULL vehicle shift: first departure -> current/final depot return.
        if first_departure is not None and return_time - first_departure > T + eps:
            result = (np.inf, False)
            return (*result, None) if return_details else result

        for j in leg:
            completions[j] = return_time
            total += return_time - rt[j]

        leg_departures.append(leg_depart)
        leg_returns.append(return_time)
        vehicle_clock = return_time

    details = {
        'arrivals': arrivals,
        'completions': completions,
        'leg_departures': leg_departures,
        'leg_returns': leg_returns,
        'first_departure': first_departure,
        'final_return': vehicle_clock if first_departure is not None else None,
        'shift_duration': (vehicle_clock - first_departure)
                          if first_departure is not None else 0.0,
    }
    return (total, True, details) if return_details else (total, True)

def truck_evaluate_solution(groups: List[RouteGroup], data_dict: dict) -> float:
    """
    Compute corrected total completion time sum_j (C[j] - r[j]).

    IMPORTANT: each RouteGroup is one vehicle and all of its legs are simulated
    sequentially on the same clock.  Depot visits between legs are replenishment
    visits, not new vehicles and not new time origins.

    Returns np.inf if any vehicle route violates a time window, depot-return
    deadline, full-shift duration, arc feasibility, or fleet-size constraint.
    """
    K = data_dict['max_vehicles']

    nonempty_groups = [g for g in groups if not g.is_empty()]
    if len(nonempty_groups) > K:
        return np.inf

    total = 0.0
    for grp in nonempty_groups:
        cost, feasible = truck_simulate_group(grp, data_dict)
        if not feasible:
            return np.inf
        total += cost

    return total

def truck_is_feasible(groups, data_dict):
    """Return True iff the solution satisfies all modelled constraints."""
    return not np.isinf(truck_evaluate_solution(groups, data_dict))

def truck_job_location(groups: List[RouteGroup]):
    """Build a reverse index: {clinic_index: (group_idx, leg_idx)}."""
    loc = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for j in leg:
                loc[j] = (gi, li)
    return loc

def truck_remove_jobs(groups: List[RouteGroup], to_remove: set) -> List[RouteGroup]:
    """
    Remove a set of clinics from the solution.

    After removal, empty legs are pruned.  Each RouteGroup retains at least one
    leg (possibly empty) so the group structure (vehicle count) is preserved for
    subsequent repair.
    """
    new_groups = []
    for grp in groups:
        new_legs = [
            [j for j in leg if j not in to_remove]
            for leg in grp.legs
        ]
        # Keep at least one (possibly empty) leg to preserve the group slot
        new_legs = [leg for leg in new_legs if leg] or [[]]
        new_groups.append(RouteGroup(legs=new_legs))
    return new_groups

def truck_random_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Randomly remove k clinics from the solution (uniform sampling).
    Simplest destroy operator; provides exploration diversity.
    """
    all_jobs = [j for grp in groups for j in grp.all_jobs()]
    if not all_jobs:
        return [g.copy() for g in groups], []
    removed = rng.sample(all_jobs, min(k, len(all_jobs)))
    return truck_remove_jobs(groups, set(removed)), removed

def truck_worst_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Remove clinics with the largest current corrected completion contribution.

    Timing is taken from the full sequential RouteGroup simulation, so service
    and completion times of later legs include all earlier depot-return trips.
    """
    rt = data_dict['ready_times']
    costs = []

    for grp in groups:
        _, feasible, details = truck_simulate_group(grp, data_dict, return_details=True)
        if not feasible:
            continue
        for j, Cj in details['completions'].items():
            costs.append((Cj - rt[j], j))

    costs.sort(key=lambda x: -x[0] - rng.uniform(0, 1e-6))
    removed = [j for _, j in costs[:k]]
    return truck_remove_jobs(groups, set(removed)), removed

def truck_shaw_removal(groups: List[RouteGroup], data_dict, k, rng):
    """
    Shaw removal using spatial distance and ACTUAL sequential service times.
    """
    dist = data_dict['dist_matrix']
    arrival = {}

    for grp in groups:
        _, feasible, details = truck_simulate_group(grp, data_dict, return_details=True)
        if not feasible:
            continue
        arrival.update(details['arrivals'])

    all_jobs = list(arrival.keys())
    if not all_jobs:
        return [g.copy() for g in groups], []

    seed = rng.choice(all_jobs)
    removed = [seed]

    for _ in range(min(k - 1, len(all_jobs) - 1)):
        ref = rng.choice(removed)
        candidates = [j for j in all_jobs if j not in removed]
        if not candidates:
            break
        best_j = min(
            candidates,
            key=lambda j: dist[ref, j] + abs(arrival[j] - arrival[ref])
        )
        removed.append(best_j)

    return truck_remove_jobs(groups, set(removed)), removed

def truck_leg_cost(leg: List[int], data_dict: dict):
    """
    Simulate a single leg and return (return_time, depart_time, feasible).

    Kept as a simple full-resimulation helper (used by truck_evaluate_solution-style
    callers that just need a final feasibility/cost check, not incremental
    reuse). The hot path used during insertion search is truck_leg_arrival_profile
    + truck_insertion_cost_fast below.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    cur_time, prev = 0.0, depot
    depart_time = None

    for node in leg:
        if np.isinf(dist[prev, node]):
            return None, None, False
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return None, None, False
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return None, None, False
    return_time = cur_time + dist[prev, depot]

    if depart_time is not None and return_time - depart_time > T + 1e-6:
        return None, None, False

    return return_time, depart_time, True

def truck_leg_arrival_profile(leg: List[int], data_dict: dict):
    """
    Full simulation of a leg that ALSO returns the per-node arrival times.

    This is the building block for the "incremental cost update" optimisation:
    profiling showed that re-simulating a leg's full timeline from the depot
    on EVERY candidate insertion position was the single largest cost in the
    ALNS main loop (~68% of total runtime, dominated by truck_leg_cost calls).

    Nodes before an insertion point are never affected by the insertion
    (time propagation is strictly left-to-right), so callers can reuse
    arrival_times[pos-1] as the starting point instead of resimulating the
    whole prefix every time.

    Returns (arrival_times, depart_time, return_time, feasible).
    arrival_times[i] is the arrival time at leg[i].
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    cur_time, prev = 0.0, depot
    depart_time = None
    arrival_times = []

    for node in leg:
        if np.isinf(dist[prev, node]):
            return None, None, None, False
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return None, None, None, False
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        arrival_times.append(arr)
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return None, None, None, False
    return_time = cur_time + dist[prev, depot]

    if depart_time is not None and return_time - depart_time > T + 1e-6:
        return None, None, None, False

    return arrival_times, depart_time, return_time, True

def truck_insertion_cost_fast(grp: RouteGroup, leg_idx: int, old_leg: List[int], pos: int,
                         j: int, data_dict: dict,
                         arr_times, depart_time_old, sum_rt_leg, return_time_old):
    """
    Incremental replacement for the old _insertion_cost.

    Reuses the leg's cached arrival profile (computed ONCE per leg by the
    caller, not once per candidate position as before) and only resimulates
    the suffix starting at `pos` — the inserted node j and everything after
    it — instead of the entire old leg AND the entire new leg from the depot.

    Because every node in a leg shares the same completion time C[j] =
    return_time, the objective delta can be computed from just
    (old return_time, new return_time, leg length, sum of ready times) in
    O(1), without re-summing per-node contributions.

    Returns (delta_obj, new_grp) or (inf, None) if insertion is infeasible.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    T     = data_dict['max_tau']
    depot = 0

    if pos == 0:
        cur_time, prev = 0.0, depot
        depart_time = None
    else:
        cur_time, prev = arr_times[pos - 1], old_leg[pos - 1]
        depart_time = depart_time_old

    for node in [j] + old_leg[pos:]:
        if np.isinf(dist[prev, node]):
            return np.inf, None
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > dd[node]:
            return np.inf, None
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        cur_time, prev = arr, node

    if np.isinf(dist[prev, depot]):
        return np.inf, None
    return_time_new = cur_time + dist[prev, depot]

    if depart_time is not None and return_time_new - depart_time > T + 1e-6:
        return np.inf, None

    old_contribution = len(old_leg) * return_time_old - sum_rt_leg
    new_contribution = (len(old_leg) + 1) * return_time_new - (sum_rt_leg + rt[j])

    new_grp = grp.copy()
    new_grp.legs[leg_idx] = old_leg[:pos] + [j] + old_leg[pos:]
    return new_contribution - old_contribution, new_grp

def truck_new_leg_cost(j: int, data_dict: dict):
    """
    Cost of opening a brand-new single-clinic leg for clinic j.

    Used when no existing leg can feasibly accommodate j (options B and C in
    truck_all_insertion_options).  The vehicle departs depot at the latest moment
    that still allows it to arrive exactly at ready_time[j].

    Returns (obj_contribution, feasible).
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    depot = 0

    if np.isinf(dist[depot, j]) or np.isinf(dist[j, depot]):
        return np.inf, False
    arr = max(dist[depot, j], rt[j])
    if arr > dd[j]:
        return np.inf, False
    return_time = arr + dist[j, depot]
    return return_time - rt[j], True

def truck_build_neighbor_lists(data_dict: dict, k: int = 20):
    """
    Candidate list: for each clinic j, precompute its k spatially nearest
    other clinics (using the arc-pruned distance matrix).

    Used to prune truck_all_insertion_options / truck_group_insertion_options: a leg
    that contains none of j's nearest neighbours is skipped outright, since
    inserting j into a leg it is geographically unrelated to is essentially
    never competitive. This cuts the number of full insertion evaluations
    from O(n) to roughly O(k) per removed clinic, without touching the
    objective/feasibility logic itself — it only restricts which candidates
    are tried, so any accepted move is exactly as valid as before.
    """
    dist = data_dict['dist_matrix']
    N    = data_dict['num_nodes']
    neighbor_lists = {}
    for j in range(1, N):
        row = dist[j]
        order = np.argsort(row)
        cands = [int(i) for i in order
                 if i != j and i != 0 and np.isfinite(row[i])]
        neighbor_lists[j] = set(cands[:k])
    return neighbor_lists

def truck_group_insertion_options(gi, grp: RouteGroup, j: int, data_dict: dict):
    """
    Enumerate feasible insertions of j into ONE RouteGroup.

    Cost is evaluated at RouteGroup level rather than leg level.  This is
    necessary for a multi-trip vehicle because changing an early leg can delay
    every later leg and can change the full-shift feasibility.

    Options:
      A) insert j at any position of an existing leg;
      B) append a new replenishment leg [j] to this same vehicle.
    """
    options = []
    neighbor_lists = data_dict.get('neighbor_lists')
    neighbors_j = neighbor_lists.get(j) if neighbor_lists else None

    old_cost, old_ok = truck_simulate_group(grp, data_dict)
    if not old_ok:
        return options

    # A) Insert into existing leg.
    for li, leg in enumerate(grp.legs):
        if leg and neighbors_j is not None and not any(n in neighbors_j for n in leg):
            continue

        for pos in range(len(leg) + 1):
            new_grp = grp.copy()
            new_grp.legs[li] = leg[:pos] + [j] + leg[pos:]
            new_cost, ok = truck_simulate_group(new_grp, data_dict)
            if not ok:
                continue
            delta = new_cost - old_cost

            def make_apply_A(gi=gi, new_grp=new_grp):
                def apply(gs):
                    gs = [g.copy() for g in gs]
                    gs[gi] = new_grp.copy()
                    return gs
                apply.gi = gi
                return apply

            options.append((delta, make_apply_A()))

    # B) Add a new trip after the existing trips of the same vehicle.
    if not grp.is_empty():
        new_grp = grp.copy()
        new_grp.legs.append([j])
        new_cost, ok = truck_simulate_group(new_grp, data_dict)
        if ok:
            delta = new_cost - old_cost

            def make_apply_B(gi=gi, new_grp=new_grp):
                def apply(gs):
                    gs = [g.copy() for g in gs]
                    gs[gi] = new_grp.copy()
                    return gs
                apply.gi = gi
                return apply

            options.append((delta, make_apply_B()))

    return options

def truck_new_group_option(groups: List[RouteGroup], j: int, data_dict: dict):
    """Option C: deploy one additional vehicle for a single-clinic trip."""
    K = data_dict['max_vehicles']
    nonempty_count = sum(not g.is_empty() for g in groups)
    if nonempty_count >= K:
        return None

    new_grp = RouteGroup(legs=[[j]])
    contrib_j, ok = truck_simulate_group(new_grp, data_dict)
    if not ok:
        return None

    def apply_C(gs, j=j):
        gs = [g.copy() for g in gs]
        gs.append(RouteGroup(legs=[[j]]))
        return gs
    return (contrib_j, apply_C)

def truck_all_insertion_options(groups: List[RouteGroup], j: int, data_dict: dict):
    """Enumerate every feasible A/B/C insertion option for clinic j."""
    options = []
    for gi, grp in enumerate(groups):
        options.extend(truck_group_insertion_options(gi, grp, j, data_dict))
    c_opt = truck_new_group_option(groups, j, data_dict)
    if c_opt is not None:
        options.append(c_opt)
    return options

def truck_greedy_insert(groups: List[RouteGroup], removed: list,
                  data_dict: dict, rng) -> List[RouteGroup]:
    """Cheapest insertion using full RouteGroup-level timing/cost evaluation."""
    groups = [g.copy() for g in groups]
    removed = list(removed)
    rng.shuffle(removed)

    for j in removed:
        opts = truck_all_insertion_options(groups, j, data_dict)
        if not opts:
            # No feasible insertion under the current partial solution.
            # Return an intentionally infeasible candidate; the ALNS caller will
            # reject it rather than silently exceeding the fleet/shift limits.
            groups.append(RouteGroup(legs=[[j]]))
            continue
        opts.sort(key=lambda x: x[0])
        _, apply = opts[0]
        groups = apply(groups)

    return [g for g in groups if not g.is_empty()]

def truck_regret2_insert(groups: List[RouteGroup], removed: list,
                   data_dict: dict, rng) -> List[RouteGroup]:
    """
    Regret-2 insertion.  Candidate costs are recomputed from full RouteGroup
    schedules so downstream legs and full-shift duration remain correct.
    """
    groups = [g.copy() for g in groups]
    unplaced = list(removed)

    while unplaced:
        best_j = None
        best_regret = -np.inf
        best_apply = None

        for j in unplaced:
            opts = truck_all_insertion_options(groups, j, data_dict)
            if not opts:
                regret = -np.inf
                apply_fn = None
            else:
                opts.sort(key=lambda x: x[0])
                apply_fn = opts[0][1]
                regret = (opts[1][0] - opts[0][0]) if len(opts) > 1 else np.inf

            if regret > best_regret:
                best_regret = regret
                best_j = j
                best_apply = apply_fn

        if best_j is None:
            # No feasible placement for any remaining request.
            # Keep the candidate structurally complete; final feasibility check
            # in the ALNS loop will reject it.
            best_j = unplaced[0]

        unplaced.remove(best_j)
        if best_apply is None:
            groups.append(RouteGroup(legs=[[best_j]]))
        else:
            groups = best_apply(groups)

    return [g for g in groups if not g.is_empty()]

class TruckAdaptiveWeights:
    """
    Roulette-wheel operator selector with segment-based weight updates.

    Each operator starts with weight 1.0.  Within each segment of
    `segment_size` iterations the operator accumulates a score based on
    the quality of solutions it produced:

        sigma[0] = reward for finding a new global best
        sigma[1] = reward for improving the current solution (not global best)
        sigma[2] = reward for a solution that is accepted (but not improving)

    At the end of each segment the weight is updated with exponential smoothing:
        w_new = decay * w_old + (1 - decay) * avg_score_this_segment

    The `decay` parameter balances long-term reputation (high decay) vs.
    responsiveness to recent performance (low decay).

    Selection uses roulette wheel (fitness-proportionate sampling):
    an operator's probability of being selected equals its weight divided by
    the total weight of all operators.
    """
    def __init__(self, operators, sigma1=3, sigma2=2, sigma3=1,
                 decay=0.8, segment_size=100):
        self.operators    = operators
        self.sigma        = [sigma1, sigma2, sigma3]
        self.decay        = decay
        self.segment_size = segment_size
        self.weights      = {op: 1.0 for op in operators}
        self._scores      = {op: 0.0 for op in operators}
        self._uses        = {op: 0   for op in operators}
        self._iter        = 0

    def select(self, rng):
        """Sample one operator proportional to its current weight."""
        ops   = self.operators
        w     = [self.weights[op] for op in ops]
        total = sum(w)
        r     = rng.uniform(0, total)
        cum   = 0.0
        for op, wi in zip(ops, w):
            cum += wi
            if r <= cum:
                return op
        return ops[-1]   # fallback for floating-point edge cases

    def update(self, op, reward_tier):
        """
        Record that `op` was used and produced a result of `reward_tier`:
            0 = new global best
            1 = improved current solution (but not global best)
            2 = accepted a worse solution (or rejected)
        """
        self._scores[op] += self.sigma[reward_tier]
        self._uses[op]   += 1
        self._iter       += 1
        if self._iter % self.segment_size == 0:
            self._flush_weights()

    def _flush_weights(self):
        """Apply exponential smoothing at the end of each segment and reset accumulators."""
        for op in self.operators:
            if self._uses[op] > 0:
                score = self._scores[op] / self._uses[op]
                self.weights[op] = (self.decay * self.weights[op] +
                                    (1 - self.decay) * score)
            self._scores[op] = 0.0
            self._uses[op]   = 0

def truck_build_initial_solution(data_dict: dict):
    """
    Construct the ALNS starting solution using the greedy nearest-neighbour heuristic.

    Each greedy route becomes a RouteGroup with a single leg (no replenishment).
    Clinics that the greedy could not assign ('unvisited') are each placed in their
    own single-clinic RouteGroup, provided the direct depot→j→depot trip is feasible.
    This ensures the initial solution is complete (all clinics assigned), which is
    required for the ALNS main loop to operate correctly.
    """
    routes_raw, _, unvisited = truck_greedy_nearest_neighbor(data_dict)
    groups = [RouteGroup(legs=[list(r)]) for r, _ in routes_raw]

    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    dd    = data_dict['due_dates']
    depot = 0
    for j in unvisited:
        if not np.isinf(dist[depot, j]) and not np.isinf(dist[j, depot]):
            arr = max(dist[depot, j], rt[j])
            if arr <= dd[j]:
                groups.append(RouteGroup(legs=[[j]]))

    return groups

def truck_request_contributions(groups, data_dict):
    """
    Return (request, corrected completion contribution, group index), sorted
    from highest to lowest contribution.
    """
    rt = data_dict['ready_times']
    scored = []

    for gi, grp in enumerate(groups):
        cost, feasible, details = truck_simulate_group(grp, data_dict, return_details=True)
        if not feasible or details is None:
            continue

        for j, completion in details['completions'].items():
            scored.append((int(j), float(completion - rt[j]), int(gi)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

def truck_select_request_induced_groups(groups, data_dict,
                                   q_requests=8,
                                   min_groups=3,
                                   max_groups=5):
    """
    Select high-cost requests first.  The vehicles containing those requests
    define the restricted exact neighborhood.
    """
    ranked = truck_request_contributions(groups, data_dict)

    selected_requests = []
    affected_groups = []

    for j, score, gi in ranked:
        if gi not in affected_groups and len(affected_groups) >= max_groups:
            continue

        selected_requests.append(j)
        if gi not in affected_groups:
            affected_groups.append(gi)

        if len(selected_requests) >= q_requests and len(affected_groups) >= min_groups:
            break

    if len(affected_groups) < min_groups:
        for j, score, gi in ranked:
            if gi not in affected_groups:
                affected_groups.append(gi)
                if j not in selected_requests:
                    selected_requests.append(j)
            if len(affected_groups) >= min_groups:
                break

    return selected_requests, affected_groups[:max_groups], ranked

def truck_request_based_mip_local_search(groups, data_dict,
                                   q_requests=8,
                                   min_groups=3,
                                   max_groups=5,
                                   time_limit=20,
                                   threads=4,
                                   verbose=False):
    """
    Jointly reoptimize the vehicle groups induced by the worst requests.

    The subproblem is solved with the same strict Rocha Gurobi formulation used
    for the full exact benchmark.  A candidate replaces the incumbent only when
    the complete ALNS objective strictly improves.
    """
    current_total = truck_evaluate_solution(groups, data_dict)

    selected_requests, affected, ranked = truck_select_request_induced_groups(
        groups, data_dict,
        q_requests=q_requests,
        min_groups=min_groups,
        max_groups=max_groups
    )

    info = {
        'selected_requests': selected_requests,
        'affected_groups': affected,
        'sub_jobs': [],
        'mip_status': 'Skipped',
        'mip_obj': None,
        'candidate_obj': current_total,
        'improved': False,
        'mip_time': 0.0,
    }

    if not affected:
        return groups, current_total, info

    sub_jobs = []
    for gi in affected:
        sub_jobs.extend(groups[gi].all_jobs())
    sub_jobs = list(dict.fromkeys(sub_jobs))
    info['sub_jobs'] = sub_jobs

    sub_nodes = [0] + sub_jobs
    sub_data = {
        'coords': np.asarray(data_dict['coords'])[sub_nodes],
        'ready_times': np.asarray(data_dict['ready_times'])[sub_nodes],
        'due_dates': np.asarray(data_dict['due_dates'])[sub_nodes],
        'max_vehicles': len(affected),
        'max_tau': data_dict['max_tau'],
        'num_nodes': len(sub_nodes),
        'dist_matrix': np.asarray(data_dict['dist_matrix'])[np.ix_(sub_nodes, sub_nodes)],
    }

    t0 = time.time()
    mip_res = truck_mip_build_and_solve(
        sub_data,
        time_limit=time_limit,
        solver_name='GUROBI',
        threads=threads,
        verbose=False,
        use_warm_start=False
    )
    info['mip_time'] = time.time() - t0
    info['mip_status'] = mip_res.get('status')
    info['mip_obj'] = mip_res.get('objective')

    mip_groups = mip_res.get('route_groups')
    if not mip_groups:
        return groups, current_total, info

    remapped_groups = []
    for vehicle_legs in mip_groups:
        mapped_legs = []
        for leg in vehicle_legs:
            mapped_legs.append([sub_nodes[k] for k in leg])
        remapped_groups.append(RouteGroup(legs=mapped_legs))

    while len(remapped_groups) < len(affected):
        remapped_groups.append(RouteGroup(legs=[]))

    if len(remapped_groups) > len(affected):
        info['mip_status'] = 'Rejected: too many route groups'
        return groups, current_total, info

    candidate = [g.copy() for g in groups]
    for slot, gi in enumerate(affected):
        candidate[gi] = remapped_groups[slot]

    candidate_obj = truck_evaluate_solution(candidate, data_dict)
    info['candidate_obj'] = candidate_obj
    info['improved'] = (
        np.isfinite(candidate_obj)
        and candidate_obj < current_total - 1e-6
    )

    if verbose:
        print(
            f"  [Restricted MIP] groups={affected}, jobs={len(sub_jobs)}, "
            f"{current_total:.4f} -> {candidate_obj:.4f}, "
            f"status={info['mip_status']}, time={info['mip_time']:.2f}s"
        )

    if info['improved']:
        return candidate, candidate_obj, info

    return groups, current_total, info

def truck_alns_build_and_solve(data_dict,
               max_iter=2000,
               time_limit=300,
               removal_min=2,
               removal_max=6,
               t_init=None,
               cooling=0.9975,
               segment_size=100,
               seed=42,
               verbose=True,
               mip_local_freq=0,
               mip_local_time=20,
               mip_local_q=8,
               mip_local_min_groups=3,
               mip_local_max_groups=5,
               candidate_list_size=20):
    """
    Main ALNS solver for CDSP with Simulated Annealing acceptance and optional
    MIP local search refinement.

    Algorithm flow
    --------------
    1. Build initial solution with greedy nearest-neighbour.
    2. For each iteration (up to max_iter or time_limit):
       a. Optionally run request-based restricted MIP on the current best solution.
       b. Select destroy operator d_op (roulette wheel, adaptive weights).
       c. Select repair operator r_op (roulette wheel, adaptive weights).
       d. Remove k clinics (k sampled uniformly from [removal_min, removal_max],
          capped at N//3 to avoid destroying too much on large instances).
       e. Re-insert removed clinics with r_op.
       f. Verify all clinics are present (skip if repair was incomplete).
       g. Accept/reject the candidate via Simulated Annealing:
            accepted if delta < 0  OR  random() < exp(−delta / T)
       h. Update operator scores and weights.
       i. Cool the temperature: T *= cooling.
    3. Return best solution found.

    Simulated Annealing temperature
    --------------------------------
    t_init defaults to max(greedy_obj * 0.05, 1.0), calibrated so that a 5%
    deterioration is accepted ~37% of the time at the start.  The cooling factor
    is applied every iteration; at cooling=0.9975 the temperature halves every
    ~277 iterations.

    Reward tiers for operator weight updates
    -----------------------------------------
    0 (sigma1) : candidate is a new global best
    1 (sigma2) : candidate improves current solution but not global best
    2 (sigma3) : candidate is accepted (worse or equal) or rejected

    Parameters
    ----------
    data_dict        : preprocessed instance dict
    max_iter         : maximum number of ALNS iterations
    time_limit       : wall-clock cutoff in seconds
    removal_min/max  : bounds on k (number of clinics removed per iteration)
    t_init           : initial SA temperature (None = auto)
    cooling          : multiplicative cooling factor per iteration
    segment_size     : number of iterations per adaptive weight update segment
    seed             : RNG seed for reproducibility
    verbose          : print progress every 200 iterations
    mip_local_freq   : run MIP local search every this many iterations (0 = disabled)
    mip_local_time   : time budget (seconds) per MIP local search call
    mip_local_q       : number of high-cost requests used to induce the exact neighborhood
    mip_local_min_groups : minimum number of affected vehicle groups
    mip_local_max_groups : maximum number of affected vehicle groups
    candidate_list_size : size of the spatial candidate list used to prune
                          insertion search (see truck_build_neighbor_lists);
                          set to a large number (>= num_nodes) to disable
                          pruning and fall back to exhaustive search.

    Returns
    -------
    dict: objective, routes (flat list of legs), groups (List[RouteGroup]),
          n_replenishments, wall_time, iterations,
          destroy_weights, repair_weights
    """
    rng = random.Random(seed)
    t0  = time.time()

    # Candidate list: precomputed once per instance, reused by every destroy/
    # repair call via data_dict['neighbor_lists']. See truck_build_neighbor_lists.
    data_dict['neighbor_lists'] = truck_build_neighbor_lists(data_dict, k=candidate_list_size)

    current_groups = truck_build_initial_solution(data_dict)
    current_obj    = truck_evaluate_solution(current_groups, data_dict)
    best_groups    = [g.copy() for g in current_groups]
    best_obj       = current_obj

    if t_init is None:
        t_init = max(current_obj * 0.05, 1.0)
    temperature = t_init

    if verbose:
        print(f"ALNS start  | Greedy obj (corrected) = {current_obj:.4f}")
        print(f"            | t_init={t_init:.4f}  cooling={cooling}  "
              f"max_iter={max_iter}  time_limit={time_limit}s")
        if mip_local_freq > 0:
            print(f"            | Restricted MIP: every {mip_local_freq} iters, "
                  f"q={mip_local_q}, groups={mip_local_min_groups}-{mip_local_max_groups}, "
                  f"{mip_local_time}s budget")

    destroy_ops = ['random', 'worst', 'shaw']
    repair_ops  = ['greedy', 'regret2']
    d_weights   = TruckAdaptiveWeights(destroy_ops, segment_size=segment_size)
    r_weights   = TruckAdaptiveWeights(repair_ops,  segment_size=segment_size)
    destroy_fn  = {'random': truck_random_removal, 'worst': truck_worst_removal, 'shaw': truck_shaw_removal}
    repair_fn   = {'greedy': truck_greedy_insert,  'regret2': truck_regret2_insert}

    # Diagnostics only; these counters do not affect the search.
    destroy_usage = {op: 0 for op in destroy_ops}
    repair_usage = {op: 0 for op in repair_ops}
    destroy_new_best = {op: 0 for op in destroy_ops}
    repair_new_best = {op: 0 for op in repair_ops}
    destroy_improve_current = {op: 0 for op in destroy_ops}
    repair_improve_current = {op: 0 for op in repair_ops}
    mip_details = []

    for it in range(1, max_iter + 1):
        if time.time() - t0 > time_limit:
            break

        # ── Optional request-based restricted exact intensification ───────────
        if mip_local_freq > 0 and it % mip_local_freq == 0 and best_groups:
            improved_groups, improved_obj, mip_info = truck_request_based_mip_local_search(
                best_groups,
                data_dict,
                q_requests=mip_local_q,
                min_groups=mip_local_min_groups,
                max_groups=mip_local_max_groups,
                time_limit=mip_local_time,
                threads=4,
                verbose=verbose
            )
            mip_info['iteration'] = it
            mip_info['before_best'] = best_obj
            mip_info['after_best'] = improved_obj
            mip_details.append(mip_info)

            if improved_obj < best_obj - 1e-6:
                best_groups = current_groups = [g.copy() for g in improved_groups]
                best_obj = current_obj = improved_obj
                if verbose:
                    print(f"  [MIP] Iter {it}: new best = {best_obj:.4f}")

        # ── Select operators ──────────────────────────────────────────────────
        d_op = d_weights.select(rng)
        r_op = r_weights.select(rng)
        destroy_usage[d_op] += 1
        repair_usage[r_op] += 1

        # Removal size k: scale with instance size, capped to avoid over-destruction
        N = data_dict['num_nodes'] - 1
        k = rng.randint(removal_min, min(removal_max, max(removal_min, N // 3)))

        # ── Destroy → Repair ──────────────────────────────────────────────────
        partial, removed = destroy_fn[d_op](current_groups, data_dict, k, rng)
        if not removed:
            continue
        candidate_groups = repair_fn[r_op](partial, removed, data_dict, rng)

        # Sanity check: all clinics must be present after repair
        expected = set(range(1, data_dict['num_nodes']))
        if set(j for g in candidate_groups for j in g.all_jobs()) != expected:
            continue

        cand_obj = truck_evaluate_solution(candidate_groups, data_dict)
        if np.isinf(cand_obj):
            continue

        # ── Simulated Annealing acceptance criterion ──────────────────────────
        delta    = cand_obj - current_obj
        accepted = (delta < 0) or (
            temperature > 1e-10 and rng.random() < math.exp(-delta / temperature)
        )

        # ── Determine reward tier ─────────────────────────────────────────────
        if cand_obj < best_obj:
            reward_tier = 0          # new global best
            destroy_new_best[d_op] += 1
            repair_new_best[r_op] += 1
            best_groups = [g.copy() for g in candidate_groups]
            best_obj    = cand_obj
        elif accepted and delta < 0:
            reward_tier = 1          # improved current but not global best
            destroy_improve_current[d_op] += 1
            repair_improve_current[r_op] += 1
        else:
            reward_tier = 2          # accepted worse or rejected

        if accepted:
            current_groups = candidate_groups
            current_obj    = cand_obj

        d_weights.update(d_op, reward_tier)
        r_weights.update(r_op, reward_tier)
        temperature *= cooling

        if verbose and it % 200 == 0:
            elapsed = time.time() - t0
            print(f"  Iter {it:5d} | best={best_obj:.4f}  "
                  f"current={current_obj:.4f}  T={temperature:.4f}  "
                  f"time={elapsed:.1f}s")

    wall_time = time.time() - t0
    if verbose:
        print(f"\nALNS done   | Best corrected obj = {best_obj:.4f}  "
              f"({wall_time:.2f}s, {it} iters)")
        print(f"  Destroy weights : { {op: round(d_weights.weights[op],3) for op in destroy_ops} }")
        print(f"  Repair  weights : { {op: round(r_weights.weights[op],3) for op in repair_ops} }")

    flat_routes = [leg for g in best_groups for leg in g.legs if leg]
    return {
        'objective'        : round(best_obj, 4),
        'routes'           : flat_routes,
        'groups'           : best_groups,
        'n_replenishments' : sum(g.n_replenishments() for g in best_groups),
        'wall_time'        : round(time.time() - t0, 2),
        'iterations'       : it,
        'destroy_weights'  : {op: round(d_weights.weights[op], 4) for op in destroy_ops},
        'repair_weights'   : {op: round(r_weights.weights[op], 4) for op in repair_ops},
    }



## 2. Greedy Heuristic

Builds the initial solution in two phases:
1. Nearest-neighbour truck routes covering all customers.
2. For each truck customer, try to convert it to a drone sortie if the delta is negative.


In [21]:
def _leg_cost_truck(leg: List[int], data_dict: dict) -> Optional[float]:
    """Return truck-only return_time for a leg, or None if infeasible."""
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    due   = data_dict['due_dates']
    T_max = data_dict['max_tau']
    depot = 0
    cur_time, prev, depart_time = 0.0, depot, None
    for node in leg:
        if np.isinf(dist[prev, node]):
            return None
        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > due[node] + 1e-9:
            return None
        if depart_time is None:
            depart_time = arr - dist[depot, node]
        cur_time, prev = arr, node
    if np.isinf(dist[prev, depot]):
        return None
    return_time = cur_time + dist[prev, depot]
    if depart_time is not None and return_time - depart_time > T_max + 1e-6:
        return None
    return return_time


def _nn_truck_routes(data_dict: dict) -> List[RouteGroup]:
    """
    Build truck-only routes by nearest-neighbour (same logic as only_Vehicle).
    Stops after K vehicles; any remaining unvisited clinics are handled by
    build_initial_solution as single-node fallback groups.
    Returns List[RouteGroup] with single-leg groups.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    due   = data_dict['due_dates']
    K     = data_dict['max_vehicles']
    T     = data_dict['max_tau']
    N     = data_dict['num_nodes']
    depot = 0

    unvisited = set(range(1, N))
    groups = []

    for _ in range(K):
        if not unvisited:
            break
        route, cur_time, prev = [], 0.0, depot
        depart_time = None

        while unvisited:
            best_j, best_arr = None, np.inf
            for j in unvisited:
                if np.isinf(dist[prev, j]):
                    continue
                arr = max(cur_time + dist[prev, j], rt[j])
                if arr > due[j] + 1e-9:
                    continue
                ret = arr + dist[j, depot]
                effective_depart = depart_time if depart_time is not None else (arr - dist[depot, j])
                if ret - effective_depart > T + 1e-6:
                    continue
                if arr < best_arr:
                    best_j, best_arr = j, arr
            if best_j is None:
                break
            route.append(best_j)
            if depart_time is None:
                depart_time = best_arr - dist[depot, best_j]
            cur_time = best_arr
            prev = best_j
            unvisited.discard(best_j)

        if route:
            groups.append(RouteGroup(legs=[route]))

    return _try_split_legs(groups, data_dict)


def _find_best_sortie(j: int,
                      leg: List[int],
                      active_sorties: List[Tuple],
                      data_dict: dict,
                      sortie_idx: Dict,
                      max_candidates: int = 20) -> Optional[Tuple]:
    """
    Find the best sortie ⟨i, j, k⟩ that reduces the leg's objective contribution.

    k can be a truck customer node (k-in-leg) or 0 (depot).

    Delta formula (exact, uses _simulate_leg_vrpd_full):
        old_contrib = old_rt × n_truck − Σ rt[truck] + Σ (drone_at_q − rt[q])
        new_contrib = new_rt × (n_truck−1) − Σ rt[truck\j] + Σ (new_drone_at_q − rt[q])
                    + (drone_at_j_new − rt[j])       ← j is now a drone customer
        delta       = new_contrib − old_contrib
    """
    if j not in sortie_idx:
        return None

    # Guard: j cannot become a drone-delivery node if it is already acting
    # as a launch node (s[0]==j) or recovery node (s[2]==j) for an existing
    # sortie in this leg.  Converting it would invalidate those sorties and
    # make evaluate_solution_vrpd return inf (i/k no longer in node_pos).
    if any(s[0] == j or s[2] == j for s in active_sorties):
        return None

    leg_without_j = [n for n in leg if n != j]
    leg_set       = set(leg_without_j)
    rt_vals       = data_dict['ready_times']
    dd            = data_dict['dist_drone']

    old_full = _simulate_leg_vrpd_full(leg, active_sorties, data_dict)
    if old_full is None:
        return None
    old_rt, old_drones = old_full

    n_truck   = len(leg)
    old_contrib = (old_rt * n_truck
                   - sum(rt_vals[v] for v in leg)
                   + sum(old_drones.get(s[1], old_rt) - rt_vals[s[1]]
                         for s in active_sorties))

    # ── Candidates: k-in-leg and k=depot ─────────────────────────────────────
    cand_reg = [
        s for s in sortie_idx[j]
        if s[2] != 0
        and s[0] in leg_set and s[2] in leg_set
        and leg_without_j.index(s[0]) < leg_without_j.index(s[2])
    ]
    cand_k0  = [
        s for s in sortie_idx[j]
        if s[2] == 0 and s[0] in leg_set
    ]
    candidates = (cand_reg + cand_k0)
    candidates.sort(key=lambda s: dd[s[0], s[1]] + dd[s[1], s[2]])
    candidates = candidates[:max_candidates]

    best_delta, best_sortie = 0.0, None

    for s in candidates:
        new_full = _simulate_leg_vrpd_full(
            leg_without_j, active_sorties + [s], data_dict)
        if new_full is None:
            continue
        new_rt, new_drones = new_full

        new_contrib = (new_rt * (n_truck - 1)
                       - sum(rt_vals[v] for v in leg if v != j)
                       + sum(new_drones.get(s2[1], new_rt) - rt_vals[s2[1]]
                             for s2 in active_sorties)
                       + new_drones.get(j, new_rt) - rt_vals[j])

        delta = new_contrib - old_contrib
        if delta < best_delta:
            best_delta  = delta
            best_sortie = s

    return (best_delta, best_sortie) if best_sortie else None





def _try_split_legs(groups: List[RouteGroup], data_dict: dict) -> List[RouteGroup]:
    """
    Try replenishment splits, but accept a split only if the COMPLETE vehicle
    group remains feasible on the continuous shift clock.

    This replaces the old independent-leg check, which could create several
    individually feasible legs whose combined vehicle shift exceeded max_tau.
    """
    groups = [g.copy() for g in groups]
    rt_vals = data_dict['ready_times']

    improved = True
    while improved:
        improved = False

        for gi, grp in enumerate(groups):
            for li, leg in enumerate(grp.legs):
                if len(leg) < 3:
                    continue

                # Current group objective on the continuous clock.
                base = _simulate_group_vrpd(grp, {}, gi, data_dict)
                if base is None:
                    continue
                obj0 = sum(base['completions'][j] - rt_vals[j]
                           for j in grp.all_jobs())

                best_gain = 0.0
                best_candidate = None

                for sp in range(1, len(leg)):
                    candidate = grp.copy()
                    candidate.legs[li:li+1] = [leg[:sp], leg[sp:]]

                    sim = _simulate_group_vrpd(candidate, {}, gi, data_dict)
                    if sim is None:
                        continue

                    obj_split = sum(
                        sim['completions'][j] - rt_vals[j]
                        for j in candidate.all_jobs()
                    )
                    gain = obj0 - obj_split

                    if gain > best_gain + 1e-9:
                        best_gain = gain
                        best_candidate = candidate

                if best_candidate is not None:
                    groups[gi] = best_candidate
                    improved = True
                    break

            if improved:
                break

    return groups


def build_initial_solution(data_dict: dict,
                           sortie_idx: Dict) -> Tuple[List[RouteGroup], List[Tuple]]:
    """
    Build a feasible initial solution on the SAME continuous-clock evaluator
    used by the final ALNS.

    Phase 1: construct truck routes and feasible replenishment legs.
    Phase 2: convert truck customers to drone customers only when the COMPLETE
             candidate solution remains feasible and improves the global
             objective.
    """
    groups  = _nn_truck_routes(data_dict)
    sorties: List[Tuple] = []

    # Fallback for clinics missed by the greedy truck initializer.
    covered   = {j for g in groups for j in g.all_jobs()}
    unvisited = set(range(1, data_dict['num_nodes'])) - covered
    K         = data_dict['max_vehicles']

    for j in sorted(unvisited):
        placed = False

        for gi in range(len(groups)):
            candidate = [g.copy() for g in groups]
            candidate[gi].legs.append([j])
            if _simulate_group_vrpd(candidate[gi], {}, gi, data_dict) is not None:
                groups = candidate
                placed = True
                break

        if not placed and len(groups) < K:
            new_grp = RouteGroup(legs=[[j]])
            if _simulate_group_vrpd(new_grp, {}, len(groups), data_dict) is not None:
                groups.append(new_grp)
                placed = True

        if not placed:
            raise RuntimeError(
                f"Could not construct a feasible initial truck solution for request {j}."
            )

    # Replenishment splitting must happen before drone conversion.
    groups = _try_split_legs(groups, data_dict)

    current_obj = evaluate_solution_vrpd(groups, sorties, data_dict)
    if np.isinf(current_obj):
        raise RuntimeError("Truck initial solution is infeasible after route construction.")

    # Global-feasibility drone conversion.
    while True:
        best_obj     = current_obj
        best_groups  = None
        best_sorties = None

        for gi, grp in enumerate(groups):
            for li, leg in enumerate(grp.legs):
                if len(leg) < 2:
                    continue

                leg_set = set(leg)

                # A truck node used as launch/recovery by an existing sortie
                # cannot itself be converted into a drone customer.
                protected = set()
                for s0 in sorties:
                    protected.add(s0[0])
                    if s0[2] != 0:
                        protected.add(s0[2])

                for j in list(leg):
                    if j in protected or j not in sortie_idx:
                        continue

                    leg_without_j = [n for n in leg if n != j]
                    remain = set(leg_without_j)

                    candidates = []
                    for sortie in sortie_idx.get(j, []):
                        i, _, k = sortie

                        if i not in remain:
                            continue

                        if k == 0:
                            candidates.append(sortie)
                        elif (k in remain and
                              leg_without_j.index(i) < leg_without_j.index(k)):
                            candidates.append(sortie)

                    # Test the most promising geometric candidates first,
                    # but accept only after full-solution evaluation.
                    dd = data_dict['dist_drone']
                    candidates.sort(
                        key=lambda s0: dd[s0[0], s0[1]] + dd[s0[1], s0[2]]
                    )

                    for sortie in candidates[:20]:
                        cand_groups = [g.copy() for g in groups]
                        cand_groups[gi].legs[li] = [
                            n for n in cand_groups[gi].legs[li] if n != j
                        ]

                        if not cand_groups[gi].legs[li]:
                            cand_groups[gi].legs.pop(li)
                            if not cand_groups[gi].legs:
                                cand_groups[gi].legs = [[]]

                        cand_groups = [g for g in cand_groups if not g.is_empty()]
                        cand_sorties = list(sorties) + [sortie]

                        cand_obj = evaluate_solution_vrpd(
                            cand_groups, cand_sorties, data_dict
                        )

                        if cand_obj < best_obj - 1e-9:
                            best_obj     = cand_obj
                            best_groups  = cand_groups
                            best_sorties = cand_sorties

        if best_groups is None:
            break

        groups      = best_groups
        sorties     = best_sorties
        current_obj = best_obj

    return groups, sorties



<>:85: SyntaxWarning: invalid escape sequence '\j'
<>:85: SyntaxWarning: invalid escape sequence '\j'
/var/folders/3f/kqwf1bgd5t5d_6_qr6473jrw0000gn/T/ipykernel_74309/4187139676.py:85: SyntaxWarning: invalid escape sequence '\j'
  """


## 3. ALNS

### 3.1 Continuous vehicle-group simulation and objective evaluation

The evaluator uses a continuous absolute clock across all replenishment legs assigned to the same truck. A later leg cannot depart before the previous leg and any direct-to-depot drone have returned, and the full vehicle shift is limited by `max_tau`.

The problem is treated as a **medical-specimen pickup problem**. Therefore \(C_j\) is always the time the specimen reaches the laboratory:

- truck-served request: completion at the truck's depot return for that leg;
- drone sortie \((i,j,k)\) with \(k\neq0\): the specimen rejoins the truck at \(k\) and completes when that truck leg returns to the lab;
- drone sortie \((i,j,0)\): completion when the drone itself reaches the lab.


In [22]:
def _simulate_leg_vrpd_impl(
        leg: List[int],
        active_sorties: List[Tuple],
        data_dict: dict,
        earliest_departure: float = 0.0):
    """
    Simulate ONE depot-to-depot leg on an absolute clock.

    Parameters
    ----------
    earliest_departure:
        Earliest time at which this vehicle may leave the depot for this leg.
        For the first leg this is 0. For later replenishment legs it is the
        previous leg's depot return/recovery time.

    Returns
    -------
    dict or None
        {
          'depart_time': ...,
          'return_time': ...,
          'truck_completions': {j: C_j},
          'drone_completions': {j: C_j},
          'drone_at_clinic': {j: time},
        }

    Completion-time semantics
    -------------------------
    This is a PICKUP problem. C_j is the time the specimen reaches the lab.

    * truck customer j:
        C_j = return time of the leg carrying specimen j;
    * drone sortie (i,j,k), k != 0:
        the drone returns specimen j to the truck at k, so the specimen reaches
        the lab only when that truck leg returns to the depot;
    * drone sortie (i,j,0):
        the drone returns directly to the depot, so C_j is its depot-arrival
        time (before any subsequent truck waiting).
    """
    dist  = data_dict['dist_matrix']
    dd    = data_dict['dist_drone']
    rt    = data_dict['ready_times']
    due   = data_dict['due_dates']
    depot = 0
    p     = data_dict['drone_params']
    SL, SR, endurance = p['SL'], p['SR'], p['endurance']

    if not leg:
        return None

    k_in_leg = [s for s in active_sorties if s[2] != 0]
    k_depot  = [s for s in active_sorties if s[2] == 0]

    # One drone: at most one independent depot-return sortie per leg.
    if len(k_depot) > 1:
        return None

    launch_at = {}
    recovery_nodes = set()
    drone_return = {}
    drone_at_clinic = {}
    depot_drone_arrivals = {}
    k_depot_launch = {}

    for i, j, k in k_in_leg:
        if i in launch_at or k in recovery_nodes:
            return None
        launch_at[i] = (j, k)
        recovery_nodes.add(k)

    for i, j, _ in k_depot:
        if i in launch_at:
            return None
        k_depot_launch[i] = j

    leg_pos = {node: pos for pos, node in enumerate(leg)}

    # Launch/recovery nodes must appear in the right order.
    for i, j, k in k_in_leg:
        if i not in leg_pos or k not in leg_pos or leg_pos[i] >= leg_pos[k]:
            return None

    # Non-overlap: the next launch cannot occur before the previous recovery.
    if len(k_in_leg) > 1:
        ordered = sorted(k_in_leg, key=lambda s: leg_pos[s[0]])
        for a, b in zip(ordered, ordered[1:]):
            if leg_pos[b[0]] < leg_pos[a[2]]:
                return None

    # A depot-return sortie must be launched only after all in-leg sorties recover.
    if k_depot:
        i0 = k_depot[0][0]
        if i0 not in leg_pos:
            return None
        for s in k_in_leg:
            if leg_pos[s[2]] > leg_pos[i0]:
                return None

    # Delay departure when useful so the first request is not reached before rt.
    first = leg[0]
    if np.isinf(dist[depot, first]):
        return None
    depart_time = max(float(earliest_departure),
                      float(rt[first] - dist[depot, first]),
                      0.0)

    cur_time = depart_time
    prev = depot

    for node in leg:
        if np.isinf(dist[prev, node]):
            return None

        arr = max(cur_time + dist[prev, node], rt[node])
        if arr > due[node] + 1e-9:
            return None
        cur_time = arr

        # Recover an in-leg sortie before the truck can continue.
        if node in recovery_nodes:
            t_dr = drone_return.get(node)
            if t_dr is None:
                return None
            cur_time = max(cur_time, t_dr) + SR

        # Launch an in-leg sortie.
        if node in launch_at:
            j, k = launch_at[node]
            mission = SL + dd[node, j] + dd[j, k] + SR
            if mission > endurance + 1e-9:
                return None

            launch_time = cur_time + SL
            at_j = max(launch_time + dd[node, j], rt[j])
            if at_j > due[j] + 1e-9:
                return None

            drone_at_clinic[j] = at_j
            drone_return[k] = at_j + dd[j, k]

            # Truck spends launch service time at node i.
            cur_time = launch_time

        # Launch a direct-to-depot sortie.
        if node in k_depot_launch:
            j = k_depot_launch[node]
            mission = SL + dd[node, j] + dd[j, depot] + SR
            if mission > endurance + 1e-9:
                return None

            launch_time = cur_time + SL
            at_j = max(launch_time + dd[node, j], rt[j])
            if at_j > due[j] + 1e-9:
                return None

            drone_at_clinic[j] = at_j
            depot_drone_arrivals[j] = at_j + dd[j, depot]
            cur_time = launch_time

        prev = node

    if np.isinf(dist[prev, depot]):
        return None

    truck_return = cur_time + dist[prev, depot]

    # Vehicle/drone cannot start another trip until its depot-return drone is
    # recovered.  This makes the one-drone-per-truck constraint continuous
    # across replenishment legs.
    leg_ready_for_next = truck_return
    for _, drone_depot_time in depot_drone_arrivals.items():
        leg_ready_for_next = max(leg_ready_for_next, drone_depot_time) + SR

    truck_completions = {j: truck_return for j in leg}
    drone_completions = {}

    for i, j, k in active_sorties:
        if k == depot:
            drone_completions[j] = depot_drone_arrivals[j]
        else:
            # Specimen is transferred back to the truck at k and delivered
            # when the truck returns to the laboratory.
            drone_completions[j] = truck_return

    return {
        'depart_time': depart_time,
        'truck_return_time': truck_return,
        'return_time': leg_ready_for_next,
        'truck_completions': truck_completions,
        'drone_completions': drone_completions,
        'drone_at_clinic': drone_at_clinic,
    }


def _simulate_leg_vrpd(leg: List[int],
                       active_sorties: List[Tuple],
                       data_dict: dict,
                       earliest_departure: float = 0.0) -> Optional[float]:
    """Backward-compatible wrapper returning the time the vehicle is ready for the next leg."""
    result = _simulate_leg_vrpd_impl(
        leg, active_sorties, data_dict,
        earliest_departure=earliest_departure
    )
    return None if result is None else result['return_time']


def _simulate_leg_vrpd_full(leg: List[int],
                            active_sorties: List[Tuple],
                            data_dict: dict,
                            earliest_departure: float = 0.0):
    """
    Backward-compatible wrapper.

    Returns (return_time, completion_map) where completion_map contains the
    LAB completion times of drone customers, not their arrival time at clinic.
    """
    result = _simulate_leg_vrpd_impl(
        leg, active_sorties, data_dict,
        earliest_departure=earliest_departure
    )
    if result is None:
        return None
    return result['return_time'], result['drone_completions']


def _simulate_group_vrpd(grp: RouteGroup,
                         sorties_by_leg: Dict[Tuple[int, int], List[Tuple]],
                         group_index: int,
                         data_dict: dict):
    """
    Simulate all replenishment legs of ONE vehicle on a continuous absolute clock.

    The second leg cannot depart before the first leg (and any direct-to-depot
    drone) has returned. The full vehicle shift is measured from the first
    departure to the final ready-for-next-leg time.
    """
    T_max = float(data_dict['max_tau'])

    current_depot_time = 0.0
    first_departure = None
    completions = {}
    leg_details = []

    for li, leg in enumerate(grp.legs):
        if not leg:
            continue

        active = sorties_by_leg.get((group_index, li), [])
        result = _simulate_leg_vrpd_impl(
            leg, active, data_dict,
            earliest_departure=current_depot_time
        )
        if result is None:
            return None

        if first_departure is None:
            first_departure = result['depart_time']

        current_depot_time = result['return_time']

        completions.update(result['truck_completions'])
        completions.update(result['drone_completions'])
        leg_details.append(result)

    if first_departure is None:
        return {
            'first_departure': 0.0,
            'final_return': 0.0,
            'completions': {},
            'legs': []
        }

    if current_depot_time - first_departure > T_max + 1e-6:
        return None

    return {
        'first_departure': first_departure,
        'final_return': current_depot_time,
        'completions': completions,
        'legs': leg_details,
    }


def evaluate_solution_vrpd(groups: List[RouteGroup],
                           sorties: List[Tuple],
                           data_dict: dict) -> float:
    """
    Compute the CDSP objective sum_j (C_j - r_j).

    C_j always denotes specimen arrival at the laboratory.
    """
    N   = data_dict['num_nodes']
    K   = data_dict['max_vehicles']
    rt  = data_dict['ready_times']

    if len(groups) > K:
        return np.inf

    truck_jobs = [j for grp in groups for j in grp.all_jobs()]
    drone_jobs = [s[1] for s in sorties]

    if set(truck_jobs + drone_jobs) != set(range(1, N)):
        return np.inf
    if len(truck_jobs) != len(set(truck_jobs)):
        return np.inf
    if len(drone_jobs) != len(set(drone_jobs)):
        return np.inf
    if set(truck_jobs) & set(drone_jobs):
        return np.inf

    # Locate truck nodes.
    node_pos = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for pi, node in enumerate(leg):
                node_pos[node] = (gi, li, pi)

    # Validate sortie topology and assign sorties to a leg.
    sorties_by_leg = {}
    for s in sorties:
        i, j, k = s

        if i not in node_pos:
            return np.inf

        gi_i, li_i, pi_i = node_pos[i]

        if k == 0:
            key = (gi_i, li_i)
        else:
            if k not in node_pos:
                return np.inf
            gi_k, li_k, pi_k = node_pos[k]
            if gi_i != gi_k or li_i != li_k or pi_i >= pi_k:
                return np.inf
            key = (gi_i, li_i)

        sorties_by_leg.setdefault(key, []).append(s)

    total = 0.0
    seen_completion = {}

    for gi, grp in enumerate(groups):
        if grp.is_empty():
            continue

        result = _simulate_group_vrpd(
            grp, sorties_by_leg, gi, data_dict
        )
        if result is None:
            return np.inf

        for j, Cj in result['completions'].items():
            seen_completion[j] = Cj
            total += Cj - rt[j]

    if set(seen_completion) != set(range(1, N)):
        return np.inf

    return total


def is_feasible_vrpd(groups, sorties, data_dict) -> bool:
    return not np.isinf(evaluate_solution_vrpd(groups, sorties, data_dict))


### 3.2 Destroy operators

In [23]:
def _remove_nodes_vrpd(
        groups: List[RouteGroup],
        sorties: List[Tuple],
        to_remove: set
) -> Tuple[List[RouteGroup], List[Tuple], List[int]]:
    """
    Remove a set of truck nodes and cascade to associated sorties.

    If a launch node i OR recovery node k of a sortie is removed,
    the sortie is cancelled and its drone customer j is added to freed.

    Returns (new_groups, new_sorties, freed_nodes).
    freed_nodes = to_remove  ∪  {drone customers from cancelled sorties}.
    """
    freed = list(to_remove)
    new_sorties = []
    for s in sorties:
        i, j, k = s
        if i in to_remove or k in to_remove:
            freed.append(j)     # drone customer freed too
        else:
            new_sorties.append(s)

    new_groups = []
    for grp in groups:
        new_legs = [[c for c in leg if c not in to_remove] for leg in grp.legs]
        new_legs = [leg for leg in new_legs if leg] or [[]]
        new_groups.append(RouteGroup(legs=new_legs))

    return new_groups, new_sorties, list(set(freed))


def random_removal(groups, sorties, data_dict, k, rng):
    """Remove q random customers."""
    truck = [j for grp in groups for j in grp.all_jobs()]
    if not truck:
        return [g.copy() for g in groups], list(sorties), []
    to_remove = set(rng.sample(truck, min(k, len(truck))))
    return _remove_nodes_vrpd(groups, sorties, to_remove)


def worst_removal(groups, sorties, data_dict, k, rng):
    """Remove q customers with the highest individual objective contribution.

    FIX: now accounts for drone sortie costs.  When a truck customer j is
    the launch node i of a drone sortie (i, j_drone, k), removing j also
    cancels that sortie and returns j_drone to the freed pool.  We add the
    drone-customer contribution to j's removal cost so that high-value
    launch nodes are correctly prioritised.
    """
    rt = data_dict['ready_times']

    # Map launch-node -> drone customer (for ALL sorties, both k-in-leg and k=depot)
    launch_to_drone: Dict[int, int] = {s[0]: s[1] for s in sorties}

    costs = []
    for grp in groups:
        for leg in grp.legs:
            leg_set     = set(leg)
            leg_sorties = [s for s in sorties
                           if s[0] in leg_set and (s[2] in leg_set or s[2] == 0)]
            result = _simulate_leg_vrpd_full(leg, leg_sorties, data_dict)
            if result is None:
                return_time = _leg_cost_truck(leg, data_dict)
                if return_time is None:
                    continue
                drone_deliveries: Dict[int, float] = {}
            else:
                return_time, drone_deliveries = result

            for j in leg:
                # Truck-customer contribution
                c = return_time - rt[j]
                # If j is a drone-sortie launch node, removing j also cancels
                # that sortie → add the (soon-to-be-lost) drone saving to the
                # removal priority so we correctly re-optimise it.
                if j in launch_to_drone:
                    j_dr = launch_to_drone[j]
                    dr_time = drone_deliveries.get(j_dr, return_time)
                    c += dr_time - rt[j_dr]
                c += rng.uniform(0, 1e-6)   # tiny tie-break noise
                costs.append((c, j))

    if not costs:
        return [g.copy() for g in groups], list(sorties), []
    costs.sort(key=lambda x: -x[0])
    to_remove = set(j for _, j in costs[:k])
    return _remove_nodes_vrpd(groups, sorties, to_remove)


def shaw_removal(groups, sorties, data_dict, k, rng):
    """Remove q relatedness-clustered customers (Shaw 1998)."""
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    depot = 0
    arrival = {}
    for grp in groups:
        for leg in grp.legs:
            cur_time, prev = 0.0, depot
            for j in leg:
                arr = max(cur_time + dist[prev, j], rt[j])
                arrival[j] = arr
                cur_time, prev = arr, j

    all_jobs = list(arrival.keys())
    if not all_jobs:
        return [g.copy() for g in groups], list(sorties), []

    seed    = rng.choice(all_jobs)
    removed = [seed]
    for _ in range(min(k - 1, len(all_jobs) - 1)):
        ref  = rng.choice(removed)
        rest = [j for j in all_jobs if j not in removed]
        if not rest:
            break
        best = min(rest, key=lambda j:
                   dist[ref, j] + abs(arrival.get(j, 0) - arrival.get(ref, 0)))
        removed.append(best)

    return _remove_nodes_vrpd(groups, sorties, set(removed))


def sortie_cancel(groups, sorties, data_dict, k, rng):
    """
    Cancel k random active sorties, returning their drone customers
    to the freed pool.  Truck routes are NOT modified.

    This lets the ALNS re-assign which clinics are served by drone.
    """
    if not sorties:
        return [g.copy() for g in groups], list(sorties), []
    cancel = rng.sample(sorties, min(k, len(sorties)))
    freed  = [s[1] for s in cancel]
    new_sorties = [s for s in sorties if s not in cancel]
    return [g.copy() for g in groups], new_sorties, freed


### 3.3 Repair operators

In [24]:
def _insertion_delta_vrpd(
        grp: RouteGroup, leg_idx: int, pos: int,
        j: int,
        sorties: List[Tuple],
        data_dict: dict) -> Tuple[float, Optional[RouteGroup]]:
    """
    Cost delta of inserting truck customer j at position pos in grp.legs[leg_idx].

    Active sorties for this leg include both k-in-leg (s[0] and s[2] in leg)
    and k=depot (s[0] in leg, s[2]==0) sorties.

    Delta uses the new objective: drone customers contribute drone_at_j − rt[j].
    """
    rt_vals = data_dict['ready_times']
    old_leg = grp.legs[leg_idx]
    new_leg = old_leg[:pos] + [j] + old_leg[pos:]

    leg_nodes = set(old_leg)
    active    = [s for s in sorties
                 if s[0] in leg_nodes and (s[2] in leg_nodes or s[2] == 0)]

    old_full = _simulate_leg_vrpd_full(old_leg, active, data_dict)
    old_rt, old_drones = (old_full if old_full is not None else (0.0, {}))

    new_full = _simulate_leg_vrpd_full(new_leg, active, data_dict)
    if new_full is None:
        return np.inf, None
    new_rt, new_drones = new_full

    old_contrib = (old_rt * len(old_leg)
                   - sum(rt_vals[nd] for nd in old_leg)
                   + sum(old_drones.get(s[1], old_rt) - rt_vals[s[1]]
                         for s in active))
    new_contrib = (new_rt * len(new_leg)
                   - sum(rt_vals[nd] for nd in new_leg)
                   + sum(new_drones.get(s[1], new_rt) - rt_vals[s[1]]
                         for s in active))

    new_grp = grp.copy()
    new_grp.legs[leg_idx] = new_leg
    return new_contrib - old_contrib, new_grp


def _new_route_delta(j: int, data_dict: dict) -> Tuple[float, bool]:
    """
    Cost of opening a new single-clinic truck route for j.
    [FIX-4] Includes TW3 (shift-duration) check.
    """
    dist  = data_dict['dist_matrix']
    rt    = data_dict['ready_times']
    due   = data_dict['due_dates']
    T_max = data_dict['max_tau']
    depot = 0

    if np.isinf(dist[depot, j]) or np.isinf(dist[j, depot]):
        return np.inf, False
    arr = max(dist[depot, j], rt[j])
    if arr > due[j] + 1e-9:
        return np.inf, False

    return_time = arr + dist[j, depot]
    depart_time = arr - dist[depot, j]
    if return_time - depart_time > T_max + 1e-6:
        return np.inf, False

    return return_time - rt[j], True


def _k0_drone_delta(j: int,
                    gi: int, li: int,
                    grp: RouteGroup,
                    sorties: List[Tuple],
                    data_dict: dict,
                    sortie_idx: Dict) -> Tuple[float, Optional[Tuple]]:
    """
    Try adding freed customer j as a drone customer via a k=depot sortie
    to grp.legs[li].  j is NOT currently in this leg.

    Returns (best_delta, best_sortie) where best_delta is the marginal cost
    of serving j by drone from this leg (compared to j not being served at all).
    """
    leg      = grp.legs[li]
    leg_set  = set(leg)
    rt_vals  = data_dict['ready_times']
    dd       = data_dict['dist_drone']

    # Active sorties already assigned to this leg
    active = [s for s in sorties
              if s[0] in leg_set and (s[2] in leg_set or s[2] == 0)]

    # At most one k=depot sortie per leg
    if any(s[2] == 0 for s in active):
        return np.inf, None

    old_full = _simulate_leg_vrpd_full(leg, active, data_dict)
    if old_full is None:
        return np.inf, None
    old_rt, old_drones = old_full

    old_leg_contrib = (old_rt * len(leg)
                       - sum(rt_vals[nd] for nd in leg)
                       + sum(old_drones.get(s[1], old_rt) - rt_vals[s[1]]
                             for s in active))

    # k=depot sortie candidates for j whose launch node i is in this leg
    candidates = [s for s in sortie_idx.get(j, [])
                  if s[2] == 0 and s[0] in leg_set]
    candidates.sort(key=lambda s: dd[s[0], s[1]] + dd[s[1], 0])

    best_delta  = np.inf
    best_sortie = None

    for s in candidates[:20]:
        new_full = _simulate_leg_vrpd_full(leg, active + [s], data_dict)
        if new_full is None:
            continue
        new_rt, new_drones = new_full

        new_leg_contrib = (new_rt * len(leg)
                           - sum(rt_vals[nd] for nd in leg)
                           + sum(new_drones.get(s2[1], new_rt) - rt_vals[s2[1]]
                                 for s2 in active)
                           + new_drones.get(j, new_rt) - rt_vals[j])

        delta = new_leg_contrib - old_leg_contrib
        if delta < best_delta:
            best_delta  = delta
            best_sortie = s

    return best_delta, best_sortie


def greedy_insert_vrpd(
        groups: List[RouteGroup],
        sorties: List[Tuple],
        freed: List[int],
        data_dict: dict,
        sortie_idx: Dict,
        rng: random.Random,
        max_sortie_candidates: int = 15) -> Tuple[List[RouteGroup], List[Tuple]]:
    """
    Two-phase greedy repair.

    Phase 1 — insertion: for each freed customer j pick the cheapest of:
        (A) insert j as truck customer into an existing leg position
        (B) add j as a new replenishment leg on an existing vehicle
            (preferred over opening a new vehicle — implements replenishment arc)
        (C) add j as drone customer via k=depot sortie from an existing leg

    Phase 2 — drone conversion: scan truck customers; convert to drone
               (k-in-leg or k=depot) if it strictly reduces the objective.
    """
    K       = data_dict['max_vehicles']
    groups  = [g.copy() for g in groups]
    sorties = list(sorties)

    # ── Phase 1 ───────────────────────────────────────────────────────────────
    order = list(freed)
    rng.shuffle(order)

    for j in order:
        best_delta  = np.inf
        best_action = None

        # Option A: truck insertion into an existing leg
        for gi, grp in enumerate(groups):
            for li, leg in enumerate(grp.legs):
                for pos in range(len(leg) + 1):
                    delta, new_grp = _insertion_delta_vrpd(
                        grp, li, pos, j, sorties, data_dict)
                    if delta < best_delta:
                        best_delta = delta
                        def _make_A(gi=gi, new_grp=new_grp):
                            def action(gs, ss):
                                gs = [g.copy() for g in gs]
                                gs[gi] = new_grp
                                return gs, ss
                            return action
                        best_action = _make_A()

        # Option B: new single-customer leg.
        # Policy: replenishment arc FIRST — add the leg to the least-loaded
        # existing vehicle rather than opening a brand-new vehicle.  This keeps
        # the vehicle count down and lets the ALNS restructure multi-leg routes.
        # A genuinely new vehicle is only opened when no existing vehicle exists.
        new_cost, ok = _new_route_delta(j, data_dict)
        if ok and new_cost < best_delta:
            best_delta = new_cost
            if groups:
                # Replenishment arc: append new leg to least-loaded vehicle
                gi_min = min(range(len(groups)),
                             key=lambda x: len(groups[x].all_jobs()))
                def _make_B(gi_min=gi_min, j=j):
                    def action(gs, ss):
                        gs = [g.copy() for g in gs]
                        gs[gi_min].legs.append([j])
                        return gs, ss
                    return action
                best_action = _make_B()
            elif len(groups) < K:
                # No existing vehicles (edge case) — open a new one
                def _make_B_new(j=j):
                    def action(gs, ss):
                        gs = [g.copy() for g in gs]
                        gs.append(RouteGroup(legs=[[j]]))
                        return gs, ss
                    return action
                best_action = _make_B_new()

        # Option C: drone via k=depot sortie from an existing leg  [NEW]
        for gi, grp in enumerate(groups):
            for li in range(len(grp.legs)):
                delta_c, sortie_c = _k0_drone_delta(
                    j, gi, li, grp, sorties, data_dict, sortie_idx)
                if delta_c < best_delta:
                    best_delta = delta_c
                    def _make_D(gi=gi, sortie_c=sortie_c):
                        def action(gs, ss):
                            return [g.copy() for g in gs], ss + [sortie_c]
                        return action
                    best_action = _make_D()

        if best_action:
            groups, sorties = best_action(groups, sorties)

    groups = [g for g in groups if not g.is_empty()]

    all_truck = [j for grp in groups for j in grp.all_jobs()]
    rng.shuffle(all_truck)

    for j in all_truck:
        gi, li = None, None
        for g_i, grp in enumerate(groups):
            for l_i, leg in enumerate(grp.legs):
                if j in leg:
                    gi, li = g_i, l_i
                    break
            if gi is not None:
                break
        if gi is None or len(groups[gi].legs[li]) < 2:
            continue

        leg    = groups[gi].legs[li]
        active = [s for s in sorties
                  if s[0] in set(leg) and (s[2] in set(leg) or s[2] == 0)]

        result = _find_best_sortie(j, leg, active, data_dict, sortie_idx,
                                   max_candidates=max_sortie_candidates)
        if result is not None:
            delta, sortie = result
            if delta < -1e-9:
                groups[gi].legs[li].remove(j)
                sorties.append(sortie)
                if not groups[gi].legs[li]:
                    groups[gi].legs.pop(li)
                    if not groups[gi].legs:
                        groups[gi].legs = [[]]
                groups = [g for g in groups if not g.is_empty()]

    return groups, sorties

def _repair_options_vrpd(groups, sorties, j, data_dict, sortie_idx):
    """
    Approximate feasible insertion options for one freed request j.
    The ranking uses the same local marginal-cost calculations as the greedy
    repair; the complete candidate is still checked by evaluate_solution_vrpd
    in the ALNS main loop.
    """
    K = data_dict['max_vehicles']
    options = []

    # A) Truck insertion in existing legs
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for pos in range(len(leg) + 1):
                delta, new_grp = _insertion_delta_vrpd(
                    grp, li, pos, j, sorties, data_dict
                )
                if np.isfinite(delta) and new_grp is not None:
                    def make_action(gi=gi, new_grp=new_grp):
                        def action(gs, ss):
                            gs2 = [g.copy() for g in gs]
                            gs2[gi] = new_grp.copy()
                            return gs2, list(ss)
                        return action
                    options.append((delta, make_action()))

    # B) New replenishment leg / new vehicle
    new_cost, ok = _new_route_delta(j, data_dict)
    if ok:
        if groups:
            # Consider every existing vehicle; continuous feasibility is checked
            # later by the complete evaluator.
            for gi in range(len(groups)):
                def make_leg_action(gi=gi, j=j):
                    def action(gs, ss):
                        gs2 = [g.copy() for g in gs]
                        gs2[gi].legs.append([j])
                        return gs2, list(ss)
                    return action
                options.append((new_cost, make_leg_action()))
        elif len(groups) < K:
            def make_new_vehicle(j=j):
                def action(gs, ss):
                    gs2 = [g.copy() for g in gs]
                    gs2.append(RouteGroup(legs=[[j]]))
                    return gs2, list(ss)
                return action
            options.append((new_cost, make_new_vehicle()))

    # C) Direct-to-depot drone insertion
    for gi, grp in enumerate(groups):
        for li in range(len(grp.legs)):
            delta, sortie = _k0_drone_delta(
                j, gi, li, grp, sorties, data_dict, sortie_idx
            )
            if np.isfinite(delta) and sortie is not None:
                def make_drone_action(sortie=sortie):
                    def action(gs, ss):
                        return [g.copy() for g in gs], list(ss) + [sortie]
                    return action
                options.append((delta, make_drone_action()))

    options.sort(key=lambda x: x[0])
    return options


def regret2_insert_vrpd(
        groups: List[RouteGroup],
        sorties: List[Tuple],
        freed: List[int],
        data_dict: dict,
        sortie_idx: Dict,
        rng: random.Random) -> Tuple[List[RouteGroup], List[Tuple]]:
    """
    Regret-2 repair.

    At each step, select the request whose second-best insertion is most
    expensive relative to its best insertion, then apply its best option.
    This complements greedy repair by prioritising requests that would become
    costly to postpone.
    """
    groups = [g.copy() for g in groups]
    sorties = list(sorties)
    remaining = list(dict.fromkeys(freed))

    while remaining:
        chosen_j = None
        chosen_action = None
        chosen_regret = -np.inf
        chosen_best = np.inf

        for j in remaining:
            opts = _repair_options_vrpd(
                groups, sorties, j, data_dict, sortie_idx
            )
            if not opts:
                continue

            best_cost = opts[0][0]
            second_cost = opts[1][0] if len(opts) > 1 else best_cost + 1e6
            regret = second_cost - best_cost

            if (regret > chosen_regret + 1e-12 or
                (abs(regret - chosen_regret) <= 1e-12 and best_cost < chosen_best)):
                chosen_j = j
                chosen_action = opts[0][1]
                chosen_regret = regret
                chosen_best = best_cost

        if chosen_j is None:
            break

        groups, sorties = chosen_action(groups, sorties)
        remaining.remove(chosen_j)

    groups = [g for g in groups if not g.is_empty()]

    # Reuse the same post-repair drone-conversion phase as greedy repair.
    if groups:
        groups, sorties = greedy_insert_vrpd(
            groups, sorties, [], data_dict, sortie_idx, rng
        )

    return groups, sorties



### 3.4 Adaptive weights

In [25]:
class AdaptiveWeights:
    """
    Roulette-wheel operator selector with segment-based weight updates.

    Each operator starts with weight 1.0.  Within each segment of
    `segment_size` iterations the operator accumulates a score based on
    the quality of solutions it produced:

        sigma[0] = reward for finding a new global best
        sigma[1] = reward for improving the current solution (not global best)
        sigma[2] = reward for a solution that is accepted (but not improving)

    At the end of each segment the weight is updated with exponential smoothing:
        w_new = decay * w_old + (1 - decay) * avg_score_this_segment

    (Aligned with only_Vehicle_models_optimized.)
    """
    def __init__(self, operators, sigma1=3, sigma2=2, sigma3=1,
                 decay=0.8, segment_size=100):
        self.operators    = list(operators)
        self.sigma        = [sigma1, sigma2, sigma3]
        self.decay        = decay
        self.segment_size = segment_size
        self.weights      = {op: 1.0 for op in operators}
        self._scores      = {op: 0.0 for op in operators}
        self._uses        = {op: 0   for op in operators}
        self._iter        = 0

    def select(self, rng: random.Random):
        ops, w = self.operators, [self.weights[op] for op in self.operators]
        total  = sum(w)
        r, cum = rng.uniform(0, total), 0.0
        for op, wi in zip(ops, w):
            cum += wi
            if r <= cum:
                return op
        return ops[-1]

    def update(self, op, tier: int):
        self._scores[op] += self.sigma[tier]
        self._uses[op]   += 1
        self._iter       += 1
        if self._iter % self.segment_size == 0:
            for o in self.operators:
                if self._uses[o] > 0:
                    self.weights[o] = (self.decay * self.weights[o]
                                       + (1 - self.decay) * self._scores[o] / self._uses[o])
                self._scores[o] = 0.0
                self._uses[o]   = 0


### 3.5 Solver
The older one-route sortie-assignment MIP is retained only as an optional experimental heuristic and is disabled by default. It is not treated as the strict Rocha restricted routing MIP.

The extended ALNS is initialized from the truck-only incumbent produced with the same core search settings. Greedy and regret-2 repair are both available through adaptive roulette selection. In drone mode, `sortie_cancel` is added to the three baseline destroy operators (`random`, `worst`, `shaw`). Launch and recovery service times are set to zero (`SL = SR = 0`) to match the formulation used in the thesis.


### Final-intensification safeguard

A final best-improving single-sortie pass is now executed after the main ALNS loop. This prevents false zero-benefit cases when an expensive run terminates before reaching the scheduled 500-iteration intensification point. The pass accepts a sortie only when the complete truck–drone objective strictly improves and repeats at most five times.


In [26]:
def sortie_assignment_mip_heuristic(data_dict, grp, sortie_idx,
                        time_limit=20.0, threads=1, verbose=False):
    """
    OPTIONAL sortie-assignment heuristic for one RouteGroup.

    This is NOT the strict Rocha restricted routing MIP. It is retained only
    as an experimental drone-assignment intensification and is disabled by
    default in the final ALNS.
    Solves a bipartite matching: which truck customer i launches drone to j,
    minimising the net delta (overhead_on_i - saving_on_j) for chosen sorties.
    Returns dict: {objective_delta, sorties, proven_optimal}.
    """
    import gurobipy as gp
    from gurobipy import GRB
    from collections import defaultdict

    dist = data_dict['dist_matrix']
    dd   = data_dict['dist_drone']
    rt   = data_dict['ready_times']
    due  = data_dict['due_dates']
    p    = data_dict['drone_params']
    SL, SR = p['SL'], p['SR']

    customers = grp.all_jobs()
    if len(customers) < 2:
        return {'objective_delta': 0.0, 'sorties': [], 'proven_optimal': True}

    def arr_at(j):
        """Truck arrival at j on a solo single-customer leg from depot."""
        d0j = dist[0, j]
        return max(d0j if not np.isinf(d0j) else rt[j], rt[j])

    def cost_truck(j):
        a = arr_at(j)
        d = dist[j, 0]
        return a + (d if not np.isinf(d) else 0.0) - rt[j]

    # Enumerate feasible k=depot sorties within this group and compute delta
    candidates = []   # (i, j, delta)
    for j in customers:
        if j not in sortie_idx:
            continue
        for s in sortie_idx[j]:
            i, j2, k = s
            if k != 0 or i not in customers or i == j:
                continue
            arr_i    = arr_at(i)
            drone_j  = max(arr_i + SL + dd[i, j], rt[j])
            if drone_j > due[j] + 1e-9:
                continue          # TW2 infeasible
            d_i0 = dist[i, 0]
            if np.isinf(d_i0):
                continue
            truck_depot = arr_i + SL + d_i0
            drone_depot = drone_j + dd[j, 0]
            overhead_i  = max(truck_depot, drone_depot) + SR - (arr_i + d_i0)
            saving_j    = cost_truck(j) - (drone_j - rt[j])
            delta       = overhead_i - saving_j    # negative = beneficial
            if delta < -1e-6:
                candidates.append((i, j, delta))

    if not candidates:
        return {'objective_delta': 0.0, 'sorties': [], 'proven_optimal': True}

    m = gp.Model("cdsp_sortie_assign")
    m.setParam('OutputFlag', 1 if verbose else 0)
    m.setParam('TimeLimit', time_limit)
    m.setParam('Threads', threads)
    m.setParam('LogToConsole', 1 if verbose else 0)

    z = {(i, j): m.addVar(vtype=GRB.BINARY, obj=delta, name=f"z_{i}_{j}")
         for i, j, delta in candidates}
    m.update()

    by_i = defaultdict(list)
    by_j = defaultdict(list)
    for i, j, _ in candidates:
        by_i[i].append((i, j))
        by_j[j].append((i, j))

    # One drone per truck leg i
    for i, pairs in by_i.items():
        m.addConstr(gp.quicksum(z[p] for p in pairs) <= 1)

    # Each drone customer j served by at most one sortie
    for j, pairs in by_j.items():
        m.addConstr(gp.quicksum(z[p] for p in pairs) <= 1)

    # If a node is both a potential drone-customer and a launch node, avoid
    # using it in both roles simultaneously
    drone_nodes  = {j for _, j, _ in candidates}
    launch_nodes = {i for i, _, _ in candidates}
    for node in drone_nodes & launch_nodes:
        as_drone  = [(i, node) for i, j, _ in candidates if j == node]
        as_launch = [(node, j) for i, j, _ in candidates if i == node]
        for dp in as_drone:
            for lp in as_launch:
                if dp in z and lp in z:
                    m.addConstr(z[dp] + z[lp] <= 1)

    m.optimize()

    if m.SolCount == 0:
        return {'objective_delta': 0.0, 'sorties': [], 'proven_optimal': False}

    chosen = [(i, j, 0) for (i, j), var in z.items() if var.X > 0.5]
    return {
        'objective_delta' : m.ObjVal,
        'sorties'         : chosen,
        'proven_optimal'  : m.Status == GRB.OPTIMAL,
    }



def best_single_sortie_intensification(groups, sorties, data_dict, sortie_idx,
                                       max_per_request=12, max_total=250):
    """Try one extra drone conversion and accept only the best globally improving move."""
    current_obj = evaluate_solution_vrpd(groups, sorties, data_dict)

    node_pos = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for pi, node in enumerate(leg):
                node_pos[node] = (gi, li, pi)

    protected = set()
    for i, j, k in sorties:
        protected.add(i)
        if k != 0:
            protected.add(k)

    dd = data_dict['dist_drone']
    candidates = []

    for j, (gi, li, _) in node_pos.items():
        if j in protected or j not in sortie_idx:
            continue

        leg = list(groups[gi].legs[li])
        if len(leg) < 2:
            continue

        leg_wo_j = [n for n in leg if n != j]
        remain = set(leg_wo_j)

        local = []
        for st in sortie_idx.get(j, []):
            i, _, k = st
            if i not in remain:
                continue
            if k == 0 or (k in remain and leg_wo_j.index(i) < leg_wo_j.index(k)):
                local.append(st)

        local.sort(key=lambda st: dd[st[0], st[1]] + dd[st[1], st[2]])
        for st in local[:max_per_request]:
            candidates.append((j, gi, li, st))

    candidates = candidates[:max_total]

    best_obj = current_obj
    best_groups = None
    best_sorties = None
    best_move = None
    feasible = 0

    for j, gi, li, st in candidates:
        cand_groups = [g.copy() for g in groups]
        cand_sorties = list(sorties)

        cand_groups[gi].legs[li] = [n for n in cand_groups[gi].legs[li] if n != j]
        if not cand_groups[gi].legs[li]:
            cand_groups[gi].legs.pop(li)
            if not cand_groups[gi].legs:
                cand_groups[gi].legs = [[]]

        cand_groups = [g for g in cand_groups if not g.is_empty()]
        cand_sorties.append(st)

        z = evaluate_solution_vrpd(cand_groups, cand_sorties, data_dict)
        if not np.isfinite(z):
            continue
        feasible += 1

        if z < best_obj - 1e-9:
            best_obj = z
            best_groups = cand_groups
            best_sorties = cand_sorties
            best_move = st

    info = {
        'tested': len(candidates),
        'feasible': feasible,
        'improved': best_groups is not None,
        'before_obj': current_obj,
        'after_obj': best_obj,
        'gain': current_obj - best_obj,
        'move': best_move,
    }

    if best_groups is None:
        return [g.copy() for g in groups], list(sorties), current_obj, info

    return best_groups, best_sorties, best_obj, info


def alns_vrpd(data_dict: dict,
              sortie_idx: Dict,
              time_limit: float = 300.0,
              max_iter: int = 5000,
              removal_ratio: float = 0.15,
              removal_min: int = 2,
              removal_max: int = 40,
              t_init_factor: float = 0.004,
              no_improve_max: int = 1000,
              segment_size: int = 100,
              cooling: float = 0.9975,
              seed: int = 42,
              verbose: bool = False,
              sortie_mip_freq: int = 0,
              sortie_mip_time: float = 20.0,
              sortie_mip_routes: int = 1,
              initial_groups: Optional[List[RouteGroup]] = None,
              initial_sorties: Optional[List[Tuple]] = None,
              sortie_intensify_freq: int = 500,
              sortie_intensify_max_total: int = 250,
              sortie_intensify_max_per_request: int = 12) -> dict:
    """
    ALNS for CDSP with drones.

    Returns a result dict with keys:
        objective, groups, sorties, n_sorties, drone_customers,
        wall_time, iterations, destroy_weights, repair_weights.
    """
    rng = random.Random(seed)
    t0  = time.time()

    if initial_groups is None:
        current_groups, current_sorties = build_initial_solution(data_dict, sortie_idx)
    else:
        current_groups = [g.copy() for g in initial_groups]
        current_sorties = list(initial_sorties or [])

    current_obj = evaluate_solution_vrpd(current_groups, current_sorties, data_dict)

    if np.isinf(current_obj):
        raise RuntimeError("Initial solution is infeasible — check instance.")

    best_groups  = [g.copy() for g in current_groups]
    best_sorties = list(current_sorties)
    best_obj     = current_obj

    T_start = max(current_obj * t_init_factor, 1e-3)
    T       = T_start

    # ── Operator registry ──────────────────────────────────────────────────────
    # Truck-only mode uses the same three core destroy operators as the
    # validated baseline. Drone mode adds two extension-specific operators.
    if sortie_idx:
        destroy_ops = ['random', 'worst', 'shaw', 'sortie_cancel']
    else:
        destroy_ops = ['random', 'worst', 'shaw']

    repair_ops = ['greedy', 'regret2']
    d_weights   = AdaptiveWeights(destroy_ops, decay=0.8, segment_size=segment_size)
    r_weights   = AdaptiveWeights(repair_ops,  decay=0.8, segment_size=segment_size)
    destroy_fn = {
        'random'       : random_removal,
        'worst'        : worst_removal,
        'shaw'         : shaw_removal,
        'sortie_cancel': sortie_cancel,
}

    N_cust     = data_dict['num_nodes'] - 1
    no_improve = 0
    it         = 0
    sortie_intensification_log = []

    if verbose:
        n_dr = sum(1 for g in current_groups for j in g.all_jobs())
        print(f"ALNS-CDSP-Drone start")
        print(f"  Initial obj = {current_obj:.4f}  "
              f"({len(current_groups)} vehicles, "
              f"{len(current_sorties)} sorties, "
              f"{len(current_sorties)} drone customers)")
        print(f"  T_start={T_start:.4f}  time_limit={time_limit}s  seed={seed}")

    for it in range(1, max_iter + 1):
        elapsed = time.time() - t0
        if elapsed >= time_limit:
            break

        # Multiplicative cooling (matches only_Vehicle_models)
        T *= cooling

        # Restore best if stuck
        if no_improve >= no_improve_max:
            current_groups  = [g.copy() for g in best_groups]
            current_sorties = list(best_sorties)
            current_obj     = best_obj
            no_improve      = 0

        # ── Select operators ───────────────────────────────────────────────────
        d_op = d_weights.select(rng)
        r_op = r_weights.select(rng)

        # Removal size β (Eq. 32 of the paper)
        beta = int(min(max(removal_min, removal_ratio * N_cust), removal_max))
        beta = max(removal_min, beta + rng.randint(-min(2, beta - removal_min), 2))

        # ── Destroy ────────────────────────────────────────────────────────────
        partial_g, partial_s, freed = destroy_fn[d_op](
            current_groups, current_sorties, data_dict, beta, rng)
        if not freed:
            no_improve += 1
            continue

        # ── Repair ─────────────────────────────────────────────────────────────
        if r_op == 'regret2':
            cand_g, cand_s = regret2_insert_vrpd(
                partial_g, partial_s, freed, data_dict, sortie_idx, rng)
        else:
            cand_g, cand_s = greedy_insert_vrpd(
                partial_g, partial_s, freed, data_dict, sortie_idx, rng)

        # Sanity: all clinics must be present
        expected = set(range(1, data_dict['num_nodes']))
        served   = set(j for g in cand_g for j in g.all_jobs()) | {s[1] for s in cand_s}
        if served != expected:
            no_improve += 1
            continue

        cand_obj = evaluate_solution_vrpd(cand_g, cand_s, data_dict)
        if np.isinf(cand_obj):
            no_improve += 1
            continue


        # MIP local search: every sortie_mip_freq iters, refine sortie assignment
        # for a random sample of route groups using Gurobi bipartite matching.
        if sortie_mip_freq > 0 and it % sortie_mip_freq == 0:
            idxs = rng.sample(range(len(current_groups)),
                              min(sortie_mip_routes, len(current_groups)))
            for gi in idxs:
                try:
                    res = sortie_assignment_mip_heuristic(
                        data_dict, current_groups[gi], sortie_idx,
                        time_limit=sortie_mip_time, threads=1, verbose=False)
                    if res['objective_delta'] < -1e-6 and res['sorties']:
                        drone_set  = {s[1] for s in res['sorties']}
                        new_groups = [g.copy() for g in current_groups]
                        new_legs   = [[c for c in leg if c not in drone_set]
                                      for leg in new_groups[gi].legs]
                        new_groups[gi].legs = [l for l in new_legs if l] or [[]]
                        new_sorties = ([s for s in current_sorties
                                        if s[1] not in drone_set] + res['sorties'])
                        new_obj = evaluate_solution_vrpd(new_groups, new_sorties, data_dict)
                        if new_obj < current_obj - 1e-6:
                            current_groups  = new_groups
                            current_sorties = new_sorties
                            current_obj     = new_obj
                            if new_obj < best_obj - 1e-9:
                                best_obj     = new_obj
                                best_groups  = [g.copy() for g in new_groups]
                                best_sorties = list(new_sorties)
                                no_improve   = 0
                except Exception:
                    pass   # Gurobi not available or infeasible sub-problem

        # ── Single-sortie intensification ───────────────────────────────────────
        if (sortie_idx and sortie_intensify_freq > 0
                and it % sortie_intensify_freq == 0 and best_groups):
            new_g, new_s, new_obj, info = best_single_sortie_intensification(
                best_groups, best_sorties, data_dict, sortie_idx,
                max_per_request=sortie_intensify_max_per_request,
                max_total=sortie_intensify_max_total
            )
            info['iteration'] = it
            sortie_intensification_log.append(info)

            if info['improved'] and new_obj < best_obj - 1e-9:
                best_groups = [g.copy() for g in new_g]
                best_sorties = list(new_s)
                best_obj = new_obj
                current_groups = [g.copy() for g in new_g]
                current_sorties = list(new_s)
                current_obj = new_obj
                no_improve = 0

        # ── SA acceptance ──────────────────────────────────────────────────────
        delta    = cand_obj - current_obj
        accepted = (delta < 0) or (
            T > 1e-10 and rng.random() < math.exp(-delta / T))

        # ── Reward tier ────────────────────────────────────────────────────────
        if cand_obj < best_obj - 1e-9:
            tier = 0                                  # new global best
            best_groups  = [g.copy() for g in cand_g]
            best_sorties = list(cand_s)
            best_obj     = cand_obj
            no_improve   = 0
        elif accepted and delta < 0:
            tier = 1; no_improve = 0                  # improved current
        else:
            tier = 2                                  # accepted-worse or rejected
            no_improve += 1

        if accepted:
            current_groups  = cand_g
            current_sorties = cand_s
            current_obj     = cand_obj

        d_weights.update(d_op, tier)
        r_weights.update(r_op, tier)

        if verbose and it % 500 == 0:
            print(f"  Iter {it:6d} | best={best_obj:.4f}  "
                  f"curr={current_obj:.4f}  T={T:.5f}  "
                  f"sorties={len(best_sorties)}  "
                  f"time={elapsed:.1f}s")

    # ── Final sortie intensification pass ─────────────────────────────────────
    # Important for short/expensive runs: if the main loop terminates before a
    # scheduled 500-iteration intensification point, diagnostics may still show
    # individually improving sorties even though none were ever explicitly tested.
    #
    # Run a small deterministic best-improvement cleanup on the final incumbent.
    # Repeat only while the complete objective strictly improves.
    final_intensify_passes = 0
    while sortie_idx and final_intensify_passes < 5:
        new_g, new_s, new_obj, info = best_single_sortie_intensification(
            best_groups,
            best_sorties,
            data_dict,
            sortie_idx,
            max_per_request=sortie_intensify_max_per_request,
            max_total=sortie_intensify_max_total
        )
        info['iteration'] = it
        info['stage'] = 'final'
        sortie_intensification_log.append(info)

        if not info['improved'] or new_obj >= best_obj - 1e-9:
            break

        best_groups = [g.copy() for g in new_g]
        best_sorties = list(new_s)
        best_obj = new_obj
        final_intensify_passes += 1

    wall_time = time.time() - t0
    if verbose:
        print(f"\nALNS done | best={best_obj:.4f}  "
              f"iters={it}  time={wall_time:.2f}s")
        print(f"  Destroy weights: "
              f"{ {op: round(d_weights.weights[op], 3) for op in destroy_ops} }")
        print(f"  Best: {len(best_groups)} vehicles,  "
              f"{len(best_sorties)} drone sorties")

    return {
        'objective'      : round(best_obj, 4),
        'groups'         : best_groups,
        'sorties'        : best_sorties,
        'n_sorties'      : len(best_sorties),
        'drone_customers': [s[1] for s in best_sorties],
        'wall_time'      : round(wall_time, 2),
        'iterations'     : it,
        'destroy_weights': {op: round(d_weights.weights[op], 4) for op in destroy_ops},
        'repair_weights' : {op: round(r_weights.weights[op], 4) for op in repair_ops},
        'sortie_intensification_log': sortie_intensification_log,
    }


def solve_truck_only(*args, **kwargs):
    """Deprecated in v6. Use truck_alns_build_and_solve() for the validated baseline."""
    raise RuntimeError(
        "solve_truck_only() is deprecated in v6; use truck_alns_build_and_solve()."
    )


def compute_saving(z_vrpd: float, z_truck: float) -> float:
    """SVRP = 1 − z_ALNS / z_VRP  (as %, same as Eq. 34 of the paper)."""
    if z_truck <= 0:
        return 0.0
    return round((1 - z_vrpd / z_truck) * 100, 2)


def sortie_endurance_utilisation(sorties, data_dict) -> float:
    """Average fraction of endurance used per sortie (%)."""
    if not sorties:
        return 0.0
    dd = data_dict['dist_drone']
    p  = data_dict['drone_params']
    SL, SR, end = p['SL'], p['SR'], p['endurance']
    utils = [(SL + dd[i, j] + dd[j, k] + SR) / end for i, j, k in sorties]
    return round(float(np.mean(utils)) * 100, 1)


### Baseline alignment in v6

The truck-only comparator is now the validated final truck ALNS: continuous multi-trip timing, Random/Worst/Shaw destroy, Greedy/Regret-2 repair, and request-based restricted Rocha MIP (`q=8`, every 500 iterations, 20 s). The truck–drone ALNS starts from this truck incumbent with no active sorties, so the no-drone solution is explicitly feasible for the extended search.


## 4. Entry Point

Set `TARGET_FILES`, `SEEDS`, and `TIME_LIMIT`, then **Run All** up to this cell.
Results are printed as a summary table and stored in `results`.


In [27]:
# ============================================================================
# v7 diagnostics — explanatory only, does NOT change the optimization logic
# ============================================================================

def _truck_completion_map(groups, data_dict):
    comp = {}
    for grp in groups:
        if grp.is_empty():
            continue
        _, ok, details = truck_simulate_group(grp, data_dict, return_details=True)
        if not ok or details is None:
            return {}
        comp.update(details['completions'])
    return comp


def _drone_completion_map(groups, sorties, data_dict):
    node_pos = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for pi, node in enumerate(leg):
                node_pos[node] = (gi, li, pi)

    sorties_by_leg = {}
    for s in sorties:
        i, j, k = s
        if i not in node_pos:
            return {}
        gi_i, li_i, _ = node_pos[i]
        if k != 0:
            if k not in node_pos:
                return {}
            gi_k, li_k, _ = node_pos[k]
            if gi_i != gi_k or li_i != li_k:
                return {}
        sorties_by_leg.setdefault((gi_i, li_i), []).append(s)

    comp = {}
    for gi, grp in enumerate(groups):
        if grp.is_empty():
            continue
        sim = _simulate_group_vrpd(grp, sorties_by_leg, gi, data_dict)
        if sim is None:
            return {}
        comp.update(sim['completions'])
    return comp


def _baseline_node_positions(groups):
    pos = {}
    for gi, grp in enumerate(groups):
        for li, leg in enumerate(grp.legs):
            for pi, j in enumerate(leg):
                pos[j] = (gi, li, pi)
    return pos


def _route_compatible_sorties(groups, sortie_idx):
    positions = _baseline_node_positions(groups)
    compatible = []

    for j, candidates in sortie_idx.items():
        if j not in positions:
            continue
        gj, lj, _ = positions[j]
        leg = list(groups[gj].legs[lj])
        leg_wo_j = [n for n in leg if n != j]
        remain = set(leg_wo_j)

        for s in candidates:
            i, _, k = s
            if i not in remain:
                continue
            if k == 0:
                compatible.append(s)
            elif k in remain and leg_wo_j.index(i) < leg_wo_j.index(k):
                compatible.append(s)

    return compatible


def _try_single_sortie_from_baseline(groups, sortie, data_dict):
    i, j, k = sortie
    positions = _baseline_node_positions(groups)
    if j not in positions:
        return np.inf

    gj, lj, _ = positions[j]
    cand_groups = [g.copy() for g in groups]
    cand_groups[gj].legs[lj] = [n for n in cand_groups[gj].legs[lj] if n != j]

    if not cand_groups[gj].legs[lj]:
        cand_groups[gj].legs.pop(lj)
        if not cand_groups[gj].legs:
            cand_groups[gj].legs = [[]]

    cand_groups = [g for g in cand_groups if not g.is_empty()]
    return evaluate_solution_vrpd(cand_groups, [sortie], data_dict)


def diagnose_against_validated_baseline(data_dict, sortie_idx, res_t, res_d):
    z_t = float(res_t['objective'])
    groups_t = res_t['groups']
    groups_d = res_d['groups']
    sorties_d = res_d['sorties']

    compatible = _route_compatible_sorties(groups_t, sortie_idx)

    feasible_single = 0
    improving_single = 0
    best_single_gain = 0.0
    best_single_sortie = None

    for s in compatible:
        z = _try_single_sortie_from_baseline(groups_t, s, data_dict)
        if np.isfinite(z):
            feasible_single += 1
            gain = z_t - z
            if gain > 1e-9:
                improving_single += 1
                if gain > best_single_gain:
                    best_single_gain = float(gain)
                    best_single_sortie = s

    selected_k0 = sum(1 for s in sorties_d if s[2] == 0)
    selected_kinleg = sum(1 for s in sorties_d if s[2] != 0)

    base_comp = _truck_completion_map(groups_t, data_dict)
    final_comp = _drone_completion_map(groups_d, sorties_d, data_dict)

    selected_rows = []
    for s in sorties_d:
        i, j, k = s
        C_base = base_comp.get(j, np.nan)
        C_final = final_comp.get(j, np.nan)
        gain = C_base - C_final if np.isfinite(C_base) and np.isfinite(C_final) else np.nan
        selected_rows.append({
            'sortie': s,
            'type': 'k=0' if k == 0 else 'k!=0',
            'C_baseline': C_base,
            'C_vrpd': C_final,
            'request_completion_gain': gain,
        })

    selected_rows.sort(
        key=lambda r: (-(r['request_completion_gain'])
                       if np.isfinite(r['request_completion_gain']) else float('inf'))
    )

    return {
        'battery_feasible': int(sum(len(v) for v in sortie_idx.values())),
        'route_compatible': int(len(compatible)),
        'full_feasible_single': int(feasible_single),
        'improving_single': int(improving_single),
        'best_single_improvement': round(best_single_gain, 4),
        'best_single_sortie': best_single_sortie,
        'selected_k0': int(selected_k0),
        'selected_kinleg': int(selected_kinleg),
        'selected_rows': selected_rows,
    }


def print_drone_diagnostics(diag):
    print("  ── Drone diagnostics ─────────────────────────────────────")
    print(f"  battery-feasible sortie types : {diag['battery_feasible']}")
    print(f"  route-compatible              : {diag['route_compatible']}")
    print(f"  full-feasible single sorties  : {diag['full_feasible_single']}")
    print(f"  improving single sorties      : {diag['improving_single']}")
    print(f"  best single-sortie gain       : {diag['best_single_improvement']:.2f}")
    print(f"  selected sorties k=0 / k!=0   : {diag['selected_k0']} / {diag['selected_kinleg']}")

    if diag['best_single_sortie'] is not None:
        print(f"  best single sortie            : {diag['best_single_sortie']}")

    if diag['selected_rows']:
        print("  selected drone requests:")
        print(f"    {'sortie':>16} {'type':>6} {'C_base':>10} {'C_vrpd':>10} {'ΔC_j':>10}")
        for r in diag['selected_rows']:
            cb, cf, gg = r['C_baseline'], r['C_vrpd'], r['request_completion_gain']
            cb_s = f"{cb:.1f}" if np.isfinite(cb) else "nan"
            cf_s = f"{cf:.1f}" if np.isfinite(cf) else "nan"
            gg_s = f"{gg:.1f}" if np.isfinite(gg) else "nan"
            print(f"    {str(r['sortie']):>16} {r['type']:>6} "
                  f"{cb_s:>10} {cf_s:>10} {gg_s:>10}")
    else:
        print("  selected drone requests       : none")


### v7 explanatory diagnostics

These diagnostics are post-processing only and do not alter the optimization logic. They distinguish battery-feasible, route-compatible, fully feasible, and individually improving sorties, and report the final split between direct-to-depot (`k=0`) and truck-recovery (`k≠0`) sorties.


### v8 single-sortie intensification

Every 500 ALNS iterations, the current best solution is checked for one additional globally improving drone conversion. Each candidate is evaluated using the complete truck–drone feasibility and objective. At most one best-improving sortie is accepted per check. This changes the search strategy only, not the formulation.


### Paired baseline fix

The truck–drone comparison now uses a **route-bearing paired truck solution** as its control. `results_alns.csv` is read only to cross-check the formal vehicle-only objective; its scalar objective is never substituted for a route. On the first run, representative truck routes are generated with the formal vehicle-only configuration and saved to `paired_truck_routes_for_drone.json`. Later runs reuse exactly those routes, so the reported drone saving is attributable to the truck–drone search relative to the same starting truck solution.


### Recommended broader RQ2 experiment

Keep the 9 representative C101/R101/RC101 cases across 25, 50, and 100 requests for the size comparison. For a broader screening of **which instance structures benefit from drones**, set `EXPERIMENT_SCOPE = "full_050"`. This runs all 56 medium-size benchmark instances and therefore covers the full C/R/RC and tight/wide time-window structure at one fixed problem size. The full 168-instance experiment remains available through `full_all`, but is not necessary unless computational time is abundant.


In [ ]:

def diagnose_drone_opportunity(data, sortie_idx, groups):
    """Four-level bottleneck funnel: battery-feasible → route-compatible → arc-feasible → beneficial.
    Helps diagnose why drones are or are not being used.
    """
    dist  = data['dist_matrix']
    dd    = data['dist_drone']
    rt    = data['ready_times']
    depot = 0
    p     = data['drone_params']
    SL    = p['SL']
    end   = p['endurance']

    n_battery      = sum(len(v) for v in sortie_idx.values())
    n_route_compat = 0
    n_arc_feasible = 0
    n_cond_d       = 0
    n_k0_compat    = 0   # k=depot sub-count

    for grp in groups:
        for leg in grp.legs:
            if not leg:
                continue
            leg_set  = set(leg)
            leg_list = list(leg)

            # Simulate leg (truck-only) for arrival times
            arrival  = {}
            cur_time = 0.0
            prev     = depot
            ok       = True
            for node in leg_list:
                if np.isinf(dist[prev, node]):
                    ok = False; break
                arr = max(cur_time + dist[prev, node], rt[node])
                arrival[node] = arr
                cur_time = arr
                prev = node
            if not ok:
                continue

            for j in leg_list:
                for s in sortie_idx.get(j, []):
                    i, _, k = s

                    if k == 0:
                        if i not in leg_set:
                            continue
                        n_route_compat += 1
                        n_k0_compat    += 1
                        n_arc_feasible += 1        # no truck arc needed
                        # Feasibility: battery + TW2 (rough check, no exact timing)
                        arr_i = arrival.get(i, rt[i])
                        drone_at_j_est = max(arr_i + SL + dd[i, j], rt[j])
                        drone_mission  = SL + dd[i, j] + dd[j, 0] + p['SR']
                        if drone_mission <= end + 1e-9 and drone_at_j_est <= data['due_dates'][j] + 1e-9:
                            n_cond_d += 1

                    else:
                        if i not in leg_set or k not in leg_set:
                            continue
                        if leg_list.index(i) >= leg_list.index(k):
                            continue
                        n_route_compat += 1

                        if np.isinf(dist[i, k]):
                            continue
                        n_arc_feasible += 1

                        arr_i = arrival.get(i, rt[i])
                        if arr_i + SL + dist[i, k] > rt[k] + 1e-6:
                            n_cond_d += 1

    w = 46
    print(f"  {'── Drone bottleneck diagnosis ':─<{w}}")
    print(f"  {'(a) battery-feasible sorties':<{w-8}} {n_battery:>6}")
    print(f"  {'(b)  + route-compatible (incl. k=depot)':<{w-8}} {n_route_compat:>6}"
          f"  [{n_k0_compat} k=depot]")
    print(f"  {'(c)   + arc-feasible':<{w-8}} {n_arc_feasible:>6}")
    print(f"  {'(d)    + beneficial timing':<{w-8}} {n_cond_d:>6}")

    if n_battery == 0:
        print(f"  BOTTLENECK (a): no battery-feasible sorties → raise endurance or"
              f" check coordinate scale")
    elif n_route_compat == 0:
        print(f"  BOTTLENECK (b): no route-compatible sorties (unusual with k=depot)")
    elif n_arc_feasible == 0:
        print(f"  BOTTLENECK (c): k-in-leg arcs all inf, k=depot count = {n_k0_compat}")
    elif n_cond_d == 0:
        print(f"  BOTTLENECK (d): no beneficial timing found")
    else:
        print(f"  ✓ Structural opportunities exist — ALNS should fire sorties.")



BASE_FOLDER   = '/Users/jane/Desktop/2025-2026-2/Master Thesis/MSCDPinstances'
OUTPUT_FOLDER = '/Users/jane/Desktop/2025-2026-2/Master Thesis'

# ============================================================
# Experiment scope
# ============================================================

EXPERIMENT_SCOPE = "full_050"
# Options:
#   "representative"
#   "full_025"
#   "full_050"       <- recommended broader experiment
#   "full_100"
#   "full_all"


def build_target_files(scope):
    root = Path(BASE_FOLDER)

    if scope == "representative":
        files = [
            # C101 / R101 / RC101 are all first-series instances.
            # They are retained only as representative cross-size cases.
            root / '025' / '025_C101.txt',
            root / '050' / '050_C101.txt',
            root / '100' / '100_C101.txt',

            root / '025' / '025_R101.txt',
            root / '050' / '050_R101.txt',
            root / '100' / '100_R101.txt',

            root / '025' / '025_RC101.txt',
            root / '050' / '050_RC101.txt',
            root / '100' / '100_RC101.txt',
        ]

    elif scope in {"full_025", "full_050", "full_100"}:
        size = scope.split("_")[1]
        files = sorted((root / size).glob("*.txt"))

    elif scope == "full_all":
        files = []
        for size in ("025", "050", "100"):
            files.extend(sorted((root / size).glob("*.txt")))

    else:
        raise ValueError(
            f"Unknown EXPERIMENT_SCOPE={scope!r}. "
            "Use representative/full_025/full_050/full_100/full_all."
        )

    return sorted(f for f in files if f.exists())


TARGET_FILES = build_target_files(EXPERIMENT_SCOPE)

print(
    f"Experiment scope: {EXPERIMENT_SCOPE} | "
    f"instances: {len(TARGET_FILES)}"
)

TIME_LIMIT = 300
seed = 42  # single seed, aligned with only_Vehicle_models

DRONE_PARAMS = {
    'drone_speed_factor': 1.43,
    'endurance'         : 30.0,
    'SL'                : 0.0,
    'SR'                : 0.0,
}


# ============================================================
# PAIRED TRUCK BASELINE CONFIGURATION
# ============================================================
#
# Important:
# The truck–drone experiment must start from the SAME truck solution
# that is used as its paired control. A scalar objective from
# results_alns.csv is not enough to reconstruct the route.
#
# This cell therefore:
#   1) reads results_alns.csv only as a consistency reference;
#   2) uses exactly one paired truck solution (with routes) for VRP-D;
#   3) saves those truck routes to JSON;
#   4) reuses the saved routes automatically on later runs.
#
# The paired baseline is NEVER replaced by a CSV objective alone.
# ============================================================

import json
import pandas as pd

FORMAL_RESULTS_CSV = Path(OUTPUT_FOLDER) / "results_alns.csv"
TRUCK_ROUTE_ARCHIVE = Path(OUTPUT_FOLDER) / "paired_truck_routes_for_drone.json"

# Baseline budgets used in the formal only-vehicle experiment
FORMAL_TIME_BUDGETS = {
    "025": 60,
    "050": 120,
    "100": 300,
}

# Read formal objective results only for cross-checking.
formal_obj_lookup = {}
if FORMAL_RESULTS_CSV.exists():
    _formal_df = pd.read_csv(FORMAL_RESULTS_CSV)
    if {"file", "alns_obj"}.issubset(_formal_df.columns):
        formal_obj_lookup = dict(
            zip(_formal_df["file"].astype(str), _formal_df["alns_obj"].astype(float))
        )
        print(f"Loaded formal ALNS reference objectives from: {FORMAL_RESULTS_CSV}")
else:
    print(
        f"Formal ALNS CSV not found at {FORMAL_RESULTS_CSV}. "
        "Paired comparisons will still be valid, but the consistency cross-check is skipped."
    )

# Existing route archive, if any
if TRUCK_ROUTE_ARCHIVE.exists():
    with open(TRUCK_ROUTE_ARCHIVE, "r", encoding="utf-8") as f:
        truck_route_archive = json.load(f)
else:
    truck_route_archive = {}


def _serialize_groups(groups):
    return [
        {"legs": [[int(j) for j in leg] for leg in grp.legs]}
        for grp in groups
    ]


def _deserialize_groups(payload):
    return [
        RouteGroup(
            legs=[[int(j) for j in leg] for leg in rec["legs"]]
        )
        for rec in payload
    ]


def _save_truck_archive():
    with open(TRUCK_ROUTE_ARCHIVE, "w", encoding="utf-8") as f:
        json.dump(truck_route_archive, f, indent=2)


def get_paired_truck_baseline(fname, data):
    """
    Return a truck-only solution WITH ROUTES for the paired drone comparison.

    Priority:
      A. Reuse a previously saved route archive for this exact instance.
      B. Otherwise rerun the validated truck ALNS once using the formal
         only-vehicle parameterization, save the resulting routes, and use
         that exact solution as the drone starting point.

    The formal CSV objective is used only as a reference check because CSV
    does not contain enough information to reconstruct RouteGroup objects.
    """
    size_key = fname[:3]

    # A. Reuse exact saved routes when available
    if fname in truck_route_archive:
        rec = truck_route_archive[fname]
        groups = _deserialize_groups(rec["groups"])
        obj = evaluate_solution_vrpd(groups, [], data)

        print(
            f"  [Paired baseline] loaded saved truck routes: "
            f"obj={obj:.4f}"
        )

        return {
            "objective": float(obj),
            "groups": groups,
            "source": "saved_route_archive",
        }

    # B. First paired run: reproduce the formal only-vehicle configuration
    formal_tl = FORMAL_TIME_BUDGETS.get(size_key, 300)

    # Formal baseline code used a size-dependent removal neighborhood.
    removal_max = max(3, data["num_nodes"] // 8)

    print(
        f"  [Paired baseline] no saved route found; running truck ALNS "
        f"(time budget={formal_tl}s, removal_max={removal_max})"
    )

    res_t = truck_alns_build_and_solve(
        data,
        max_iter=5000,
        time_limit=formal_tl,
        removal_min=2,
        removal_max=removal_max,
        cooling=0.9975,
        segment_size=100,
        seed=seed,
        verbose=False,
        mip_local_freq=500,
        mip_local_time=20,
        mip_local_q=8,
        mip_local_min_groups=3,
        mip_local_max_groups=5,
        candidate_list_size=20,
    )

    # Save the exact route structure used for the subsequent drone run.
    truck_route_archive[fname] = {
        "objective": float(res_t["objective"]),
        "groups": _serialize_groups(res_t["groups"]),
        "seed": int(seed),
        "time_budget_s": float(formal_tl),
        "removal_max": int(removal_max),
    }
    _save_truck_archive()

    return {
        "objective": float(res_t["objective"]),
        "groups": [g.copy() for g in res_t["groups"]],
        "source": "fresh_formal_configuration",
    }



results = []

for fpath in TARGET_FILES:
    fname = fpath.name
    print(f"\n{'='*60}")
    print(f"File: {fname}")

    data = load_and_prepare(str(fpath), DRONE_PARAMS)
    N    = data['num_nodes'] - 1

    t_s = time.time()
    _, sortie_idx = build_sortie_set(data)
    n_sp = sum(len(v) for v in sortie_idx.values())
    print(f"  N={N}  feasible sortie types={n_sp}  "
          f"(built in {time.time()-t_s:.2f}s)")

    size_key = fname[:3]  # '025', '050', '100'

    # ------------------------------------------------------------
    # PAIRED truck-only baseline
    # ------------------------------------------------------------
    # IMPORTANT: this exact route solution is the control used for the
    # subsequent truck-drone search. Do not replace it with a scalar CSV
    # objective, because the route structure is required by VRP-D.
    res_t = get_paired_truck_baseline(fname, data)

    # Optional consistency check against the formal 100-request ALNS table.
    formal_ref_obj = formal_obj_lookup.get(fname, np.nan)
    if np.isfinite(formal_ref_obj):
        paired_diff = float(res_t['objective']) - float(formal_ref_obj)
        paired_diff_pct = (
            paired_diff / float(formal_ref_obj) * 100
            if abs(float(formal_ref_obj)) > 1e-12 else np.nan
        )
        print(
            f"  [Formal ALNS check] CSV={formal_ref_obj:.4f} | "
            f"paired route={res_t['objective']:.4f} | "
            f"diff={paired_diff:+.4f} ({paired_diff_pct:+.2f}%)"
        )
    else:
        paired_diff = np.nan
        paired_diff_pct = np.nan
    # Start the extended search from the best truck-only incumbent.
    # Therefore "no drone" is explicitly contained in the extended search and
    # the VRP-D incumbent cannot be worse simply because of initialization.
    drone_time_limit = FORMAL_TIME_BUDGETS.get(size_key, TIME_LIMIT)

    res_d = alns_vrpd(
        data, sortie_idx,
        time_limit=drone_time_limit,
        seed=seed,
        cooling=0.9975,
        sortie_mip_freq=0,
        initial_groups=res_t['groups'],
        initial_sorties=[],
        sortie_intensify_freq=500,
        sortie_intensify_max_total=250,
        sortie_intensify_max_per_request=12
    )
    # v7 diagnostics: post-processing only
    diag = diagnose_against_validated_baseline(data, sortie_idx, res_t, res_d)
    print_drone_diagnostics(diag)

    truck_objs  = [res_t['objective']]
    vrpd_objs   = [res_d['objective']]
    vrpd_n      = [res_d['n_sorties']]
    vrpd_util_list = [sortie_endurance_utilisation(res_d['sorties'], data)]

    z_truck  = float(np.mean(truck_objs))
    z_vrpd   = float(np.mean(vrpd_objs))
    saving   = compute_saving(z_vrpd, z_truck)
    avg_n    = float(np.mean(vrpd_n))
    avg_util = float(np.mean(vrpd_util_list))
    avg_saving_per_sortie = (
        (z_truck - z_vrpd) / avg_n
        if avg_n > 0 else 0.0
    )

    print(f"  [Truck-only  {1} seeds]  "
          f"obj = {z_truck:.4f}  (std={float(np.std(truck_objs)):.2f})")
    print(f"  [VRP-D       {1} seeds]  "
          f"obj = {z_vrpd:.4f}  SVRP = {saving:.1f}%  "
          f"sorties = {avg_n:.1f}  util = {avg_util:.1f}%  "
          f"(std={float(np.std(vrpd_objs)):.2f})")

    # [DIAG] Run on the INITIAL truck-only routes (before ALNS optimisation).
    # This correctly shows structural drone opportunities in the unoptimised
    # routes, rather than the (trivially empty) post-optimisation residual.
    init_groups, _ = build_initial_solution(data, {})  # empty sortie_idx → pure truck NN
    diagnose_drone_opportunity(data, sortie_idx, init_groups)

    results.append({
        'file'             : fname,
        'n_customers'      : N,
        'z_truck_avg'      : round(z_truck, 4),
        'z_truck_std'      : round(float(np.std(truck_objs)), 4),
        'z_vrpd_avg'       : round(z_vrpd, 4),
        'z_vrpd_std'       : round(float(np.std(vrpd_objs)), 4),
        'delta_obj'        : round(z_truck - z_vrpd, 2),
        'SVRP_%'           : saving,
        'avg_sorties'      : round(avg_n, 1),
        'drone_coverage_%' : round(avg_n / N * 100, 1),
        'avg_saving_per_sortie': round(avg_saving_per_sortie, 2),
        'endurance_util_%' : round(avg_util, 1),
        'formal_truck_obj_csv': (
            round(float(formal_ref_obj), 4)
            if np.isfinite(formal_ref_obj) else ''
        ),
        'paired_vs_formal_diff': (
            round(float(paired_diff), 4)
            if np.isfinite(paired_diff) else ''
        ),
        'paired_vs_formal_diff_%': (
            round(float(paired_diff_pct), 2)
            if np.isfinite(paired_diff_pct) else ''
        ),
        'paired_truck_source': res_t.get('source', ''),
        'battery_feasible_sorties': diag['battery_feasible'],
        'route_compatible_sorties': diag['route_compatible'],
        'full_feasible_single_sorties': diag['full_feasible_single'],
        'improving_single_sorties': diag['improving_single'],
        'best_single_sortie_gain': diag['best_single_improvement'],
        'selected_k0': diag['selected_k0'],
        'selected_kinleg': diag['selected_kinleg'],
        'intensify_calls': len(res_d.get('sortie_intensification_log', [])),
        'intensify_improvements': sum(
            1 for x in res_d.get('sortie_intensification_log', []) if x.get('improved')
        ),
        'intensify_total_gain': round(sum(
            x.get('gain', 0.0) for x in res_d.get('sortie_intensification_log', [])
            if x.get('improved')
        ), 4),
        'seeds'            : str([seed]),
        'time_limit_s'     : TIME_LIMIT,
    })

if not results:
    print("No files found. Check BASE_FOLDER and TARGET_FILES paths.")
else:
    out_path = Path(OUTPUT_FOLDER) / 'results_vrpd_improved.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)
    print(f"\nResults saved → {out_path}")

    print(f"\nMain performance summary:")
    hdr = (
        f"  {'File':<22} {'N':>4} {'Truck':>9} {'Truck+Drone':>11} "
        f"{'Δobj':>8} {'Saving%':>8} {'Drone%':>8} "
        f"{'Sorties':>8} {'Save/sortie':>12}"
    )
    print(hdr)
    print(f"  {'-'*103}")
    for r in results:
        print(
            f"  {r['file']:<22} {r['n_customers']:>4}"
            f" {r['z_truck_avg']:>9.1f} {r['z_vrpd_avg']:>11.1f}"
            f" {r['delta_obj']:>8.1f} {r['SVRP_%']:>7.1f}%"
            f" {r['drone_coverage_%']:>7.1f}%"
            f" {r['avg_sorties']:>8.1f}"
            f" {r['avg_saving_per_sortie']:>12.2f}"
        )

    print("\nFormal-baseline consistency check:")
    ch = (
        f"  {'File':<22} {'CSV formal':>12} {'Paired truck':>13} "
        f"{'Diff':>10} {'Diff%':>8} {'Source':>24}"
    )
    print(ch)
    print(f"  {'-'*95}")
    for r in results:
        csv_v = r.get('formal_truck_obj_csv', '')
        diff_v = r.get('paired_vs_formal_diff', '')
        diff_p = r.get('paired_vs_formal_diff_%', '')
        csv_s = f"{csv_v:.1f}" if isinstance(csv_v, (int, float)) else "-"
        diff_s = f"{diff_v:+.1f}" if isinstance(diff_v, (int, float)) else "-"
        diff_ps = f"{diff_p:+.2f}%" if isinstance(diff_p, (int, float)) else "-"
        print(
            f"  {r['file']:<22} {csv_s:>12}"
            f" {r['z_truck_avg']:>13.1f} {diff_s:>10} {diff_ps:>8}"
            f" {r.get('paired_truck_source',''):>24}"
        )

    print(f"\nDiagnostic summary:")
    dh = (f"  {'File':<22} {'Battery':>8} {'Route':>8} {'Feasible':>9} "
          f"{'Improve':>8} {'Best1':>8} {'k=0':>5} {'k!=0':>6}")
    print(dh)
    print(f"  {'-'*78}")
    for r in results:
        print(f"  {r['file']:<22}"
              f" {r['battery_feasible_sorties']:>8}"
              f" {r['route_compatible_sorties']:>8}"
              f" {r['full_feasible_single_sorties']:>9}"
              f" {r['improving_single_sorties']:>8}"
              f" {r['best_single_sortie_gain']:>8.1f}"
              f" {r['selected_k0']:>5}"
              f" {r['selected_kinleg']:>6}")

    avg_saving = np.mean([r['SVRP_%'] for r in results])
    avg_cov    = np.mean([r['drone_coverage_%'] for r in results])
    print(f"\n  Mean SVRP: {avg_saving:.1f}%   Mean drone coverage: {avg_cov:.1f}%")
    print()
    print("  Column guide:")
    print("    z_truck / z_vrpd : total Σ(C[j]-rt[j]), where C[j] is lab arrival (lower = better)")
    print("    Δobj             : z_truck - z_vrpd  (absolute time saved)")
    print("    SVRP%            : Δobj / z_truck * 100  (relative saving)")
    print("    Drone%           : sorties / N * 100  (% customers by drone)")
    print("    End.util         : avg (SL+flight+SR) / endurance  (battery used)")

Experiment scope: full_050 | instances: 56
Loaded formal ALNS reference objectives from: /Users/jane/Desktop/2025-2026-2/Master Thesis/results_alns.csv

File: 050_C101.txt
  N=50  feasible sortie types=38601  (built in 0.03s)
  [Paired baseline] loaded saved truck routes: obj=1219.4579
  ── Drone diagnostics ─────────────────────────────────────
  battery-feasible sortie types : 38601
  route-compatible              : 0
  full-feasible single sorties  : 0
  improving single sorties      : 0
  best single-sortie gain       : 0.00
  selected sorties k=0 / k!=0   : 0 / 0
  selected drone requests       : none
  [Truck-only  1 seeds]  obj = 1219.4579  (std=0.00)
  [VRP-D       1 seeds]  obj = 1219.4579  SVRP = 0.0%  sorties = 0.0  util = 0.0%  (std=0.00)
  ── Drone bottleneck diagnosis ────────────────
  (a) battery-feasible sorties            38601
  (b)  + route-compatible (incl. k=depot)    118  [58 k=depot]
  (c)   + arc-feasible                      118
  (d)    + beneficial timing   

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jane/Desktop/2025-2026-2/Master Thesis/paired_truck_routes_for_drone.json'

## 5. Detailed Insights

Run on a single file to inspect the per-sortie saving breakdown.
Set `RUN_INSIGHTS = True` and run this cell independently after section 4.


In [ ]:
RUN_INSIGHTS = False

if not RUN_INSIGHTS:
    print("Insights skipped — set RUN_INSIGHTS = False and run this cell.")
else:
    INSPECT_FILE = Path(BASE_FOLDER) / '100' / '100_R101.txt'
    INSPECT_SEED = 42
    INSPECT_TL   = 300.0

    data_ins = load_and_prepare(str(INSPECT_FILE))
    _, sidx_ins = build_sortie_set(data_ins)
    N_ins = data_ins['num_nodes'] - 1

    print(f"Running on {INSPECT_FILE.name}  (N={N_ins}) ...")
    res_d_ins = alns_vrpd(data_ins, sidx_ins, time_limit=INSPECT_TL,
                          seed=INSPECT_SEED, verbose=False)
    res_t_ins = solve_truck_only(data_ins, time_limit=INSPECT_TL,
                                 seed=INSPECT_SEED, verbose=False)

    z_t_ins  = res_t_ins['objective']
    z_d_ins  = res_d_ins['objective']
    svrp_ins = compute_saving(z_d_ins, z_t_ins)
    n_sor    = res_d_ins['n_sorties']

    print(f"\nz_truck = {z_t_ins:.1f}")
    print(f"z_vrpd  = {z_d_ins:.1f}")
    print(f"Δobj    = {z_t_ins - z_d_ins:.1f}  (= {svrp_ins:.1f}% saving)")
    print(f"Sorties = {n_sor}  ({n_sor/N_ins*100:.0f}% of {N_ins} customers by drone)")

    dist_ins = data_ins['dist_matrix']
    dd_ins   = data_ins['dist_drone']
    rt_ins   = data_ins['ready_times']
    due_ins  = data_ins['due_dates']
    sl_ins   = data_ins['drone_params']['SL']

    rows_ins = []
    for s in res_d_ins['sorties']:
        if s[2] != 0:
            continue
        i_n, j_n, _ = s
        d0i   = dist_ins[0, i_n]
        arr_i = max(d0i if not np.isinf(d0i) else rt_ins[i_n], rt_ins[i_n])
        drone_at_j = max(arr_i + sl_ins + dd_ins[i_n, j_n], rt_ins[j_n])
        C_j_drone = drone_at_j + dd_ins[j_n, 0]
        d0j   = dist_ins[0, j_n]
        arr_j_solo = max(d0j if not np.isinf(d0j) else rt_ins[j_n], rt_ins[j_n])
        dj0   = dist_ins[j_n, 0]
        C_j_truck = arr_j_solo + (dj0 if not np.isinf(dj0) else 0.0)
        saving = C_j_truck - C_j_drone
        rows_ins.append(dict(i=i_n, j=j_n,
                             rt_j=rt_ins[j_n], due_j=due_ins[j_n],
                             arr_i=arr_i, drone_at_j=drone_at_j,
                             C_j_drone=C_j_drone,
                             C_j_truck=C_j_truck, saving=saving))

    rows_ins.sort(key=lambda r: r['saving'], reverse=True)

    print(f"\nPer-sortie breakdown  (sorted by saving, top 20):")
    print(f"  {'i':>4}→{'j':<4}  {'rt[j]':>6}  {'arr_i':>7}"
          f"  {'C_drone':>8}  {'C_truck':>8}  {'saving':>8}")
    print(f"  {'-'*56}")
    for r in rows_ins[:20]:
        print(f"  {r['i']:>4}→{r['j']:<4}  {r['rt_j']:>6.0f}  {r['arr_i']:>7.1f}"
              f"  {r['C_j_drone']:>8.1f}  {r['C_j_truck']:>8.1f}  {r['saving']:>8.1f}")
    if len(rows_ins) > 20:
        print(f"  ... ({len(rows_ins)-20} more sorties not shown)")

    est_total = sum(r['saving'] for r in rows_ins)
    actual    = z_t_ins - z_d_ins
    print(f"\n  Estimated saving from {len(rows_ins)} sorties : {est_total:.1f}")
    print(f"  Actual Δobj (z_truck - z_vrpd)      : {actual:.1f}")
    print(f"  Gap (route restructuring effect)     : {actual - est_total:.1f}")

    import matplotlib.pyplot as plt

    savings_all = [r['saving'] for r in rows_ins]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Where does the {svrp_ins:.1f}% saving come from?  —  {INSPECT_FILE.name}",
                 fontsize=12, fontweight='bold')

    ax1 = axes[0]
    ax1.hist(savings_all, bins=15, color='#4A5FCF', alpha=0.85, edgecolor='white')
    ax1.axvline(np.mean(savings_all), color='#E28743', linewidth=1.8,
                linestyle='--', label=f"Mean = {np.mean(savings_all):.1f}")
    ax1.set_xlabel('Time units saved per drone sortie', fontsize=10)
    ax1.set_ylabel('Number of sorties', fontsize=10)
    ax1.set_title('Distribution of per-sortie saving', fontsize=10)
    ax1.legend(fontsize=9); ax1.grid(alpha=0.25, linestyle='--')

    ax2 = axes[1]
    cumulative = np.cumsum(sorted(savings_all, reverse=True))
    ax2.plot(range(1, len(cumulative)+1), cumulative, 'o-',
             color='#4A5FCF', linewidth=2, markersize=5)
    ax2.axhline(actual, color='#E28743', linewidth=1.5, linestyle='--',
                label=f'Actual Δobj = {actual:.0f}')
    ax2.set_xlabel('Number of sorties (ranked by saving)', fontsize=10)
    ax2.set_ylabel('Cumulative time saved', fontsize=10)
    ax2.set_title('Cumulative saving by sortie rank', fontsize=10)
    ax2.legend(fontsize=9); ax2.grid(alpha=0.25, linestyle='--')

    plt.tight_layout()
    chart_path = Path(OUTPUT_FOLDER) / 'solution_inspector.png'
    plt.savefig(str(chart_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nChart saved → {chart_path}")
    print()
    print("  C_drone = specimen arrival time at the laboratory via drone")
    print("  C_truck = specimen arrival time at the laboratory via truck")
    print("  saving  = C_truck - C_drone")


## 6. Scenario Analysis

Vehicle shortage scenario: reduce available fleet size and compare truck-only vs VRP-D.
Set `RUN_SCENARIO = True` and run this cell independently after section 4.


In [ ]:
RUN_SCENARIO = False

if not RUN_SCENARIO:
    print("Scenario skipped — set RUN_SCENARIO = False and run this cell.")
elif 'results' not in vars() or not results:
    print("Run the main loop (Cell 24) first.")
else:
    import matplotlib.pyplot as plt
    import copy as _copy

    EMRG_FILES = [
        Path(BASE_FOLDER) / '025' / '025_R101.txt',
        Path(BASE_FOLDER) / '050' / '050_R101.txt',
    ]
    EMRG_FILES   = [f for f in EMRG_FILES if f.exists()]
    URG_FRAC     = 0.30        # 30% of customers made urgent (fixed)
    URG_WINDOWS  = [15, 30, 60]  # sweep urgency tightness
    EMRG_SEEDS   = [42, 43, 44]
    EMRG_TL      = 120.0

    def create_urgent_scenario(data_orig, urgent_fraction, urgency_window, seed):
        import random as _rand
        data_mod = _copy.deepcopy(data_orig)
        N = data_mod['num_nodes']
        customers = list(range(1, N))
        n_urgent  = max(1, int(len(customers) * urgent_fraction))
        rng       = _rand.Random(seed)
        urgent    = rng.sample(customers, n_urgent)

        n_tightened = 0
        for j in urgent:
            new_due = float(data_mod['ready_times'][j]) + urgency_window
            if new_due < data_mod['due_dates'][j]:
                data_mod['due_dates'][j] = new_due
                n_tightened += 1

        data_mod = preprocess_instance(data_mod)

        # Count customers that became unreachable for truck (dist_matrix → inf)
        n_truck_infeasible = int(np.isinf(
            data_mod['dist_matrix'][0, list(urgent)]).sum())

        return data_mod, urgent, n_tightened, n_truck_infeasible

    scenario_results = []

    for fpath in EMRG_FILES:
        fname     = fpath.name
        data_base = load_and_prepare(str(fpath))
        N         = data_base['num_nodes'] - 1

        print(f"\n{'='*65}")
        print(f"File: {fname}  (N={N}, urgency fraction={URG_FRAC*100:.0f}%)")
        print(f"  {'Window':>7}  {'Tightened':>10}  {'Truck-inf':>10}"
              f"  {'z_truck':>9}  {'z_vrpd':>9}  {'SVRP%':>6}  {'Drone%':>7}")
        print(f"  {'-'*65}")

        for win in URG_WINDOWS:
            truck_objs, vrpd_objs, vrpd_ns = [], [], []
            tight_list, inf_list = [], []

            for seed in EMRG_SEEDS:
                data_s, urg, n_tight, n_inf = create_urgent_scenario(
                    data_base, URG_FRAC, win, seed)
                tight_list.append(n_tight)
                inf_list.append(n_inf)
                _, sidx = build_sortie_set(data_s)
                try:
                    rt_ = solve_truck_only(data_s, time_limit=EMRG_TL,
                                           seed=seed, verbose=False)
                    truck_objs.append(rt_['objective'])
                except RuntimeError:
                    truck_objs.append(np.inf)   # infeasible for truck-only

                try:
                    rd_ = alns_vrpd(data_s, sidx, time_limit=EMRG_TL,
                                    seed=seed, verbose=False)
                    vrpd_objs.append(rd_['objective'])
                    vrpd_ns.append(rd_['n_sorties'])
                except RuntimeError:
                    vrpd_objs.append(np.inf)
                    vrpd_ns.append(0)

            z_t  = float(np.nanmean([x for x in truck_objs if not np.isinf(x)]))                    if any(not np.isinf(x) for x in truck_objs) else np.inf
            z_d  = float(np.nanmean([x for x in vrpd_objs if not np.isinf(x)]))                    if any(not np.isinf(x) for x in vrpd_objs) else np.inf
            svrp = compute_saving(z_d, z_t) if not (np.isinf(z_t) or np.isinf(z_d)) else 0.0
            avg_n = float(np.mean(vrpd_ns))
            cov   = round(avg_n / N * 100, 1)
            avg_tight = float(np.mean(tight_list))
            avg_inf   = float(np.mean(inf_list))

            row = dict(file=fname, N=N, urg_window=win,
                       avg_tightened=round(avg_tight,1),
                       avg_truck_infeasible=round(avg_inf,1),
                       z_truck=round(z_t,2), z_vrpd=round(z_d,2),
                       delta_obj=round(z_t-z_d,2) if not np.isinf(z_t-z_d) else 0,
                       SVRP=svrp, avg_sorties=round(avg_n,1),
                       drone_coverage=cov)
            scenario_results.append(row)

            z_t_str = f"{z_t:>9.1f}" if not np.isinf(z_t) else "     inf  "
            z_d_str = f"{z_d:>9.1f}" if not np.isinf(z_d) else "     inf  "
            print(f"  {win:>6}    {avg_tight:>9.1f}  {avg_inf:>10.1f}"
                  f"  {z_t_str}  {z_d_str}  {svrp:>5.1f}%  {cov:>6.1f}%")

        print()
        print("  'Tightened': customers whose due_date was actually changed")
        print("  'Truck-inf': customers unreachable by truck after tightening")
        print("  (if Tightened≈0, urgency window is too large for this instance)")

    # ── Plot ──────────────────────────────────────────────────────────────────
    if scenario_results:
        files_in = list(dict.fromkeys(r['file'] for r in scenario_results))
        colors   = ['#4A5FCF', '#3A9E8A']

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        fig.suptitle(f"Emergency scenario analysis  (urgency fraction = {URG_FRAC*100:.0f}%)",
                     fontsize=12, fontweight='bold')

        for panel, (ax, ylabel, key, title) in enumerate(zip(axes,
            ['SVRP %', 'Drone coverage %'],
            ['SVRP',   'drone_coverage'],
            ['Objective saving vs urgency window',
             'Drone coverage vs urgency window'])):
            for i, fname in enumerate(files_in):
                rows = [r for r in scenario_results if r['file'] == fname]
                xs   = [r['urg_window'] for r in rows]
                ys   = [r[key]           for r in rows]
                ax.plot(xs, ys, 'o-', color=colors[i], linewidth=2,
                        markersize=8, label=fname.replace('.txt',''))
            ax.set_xlabel('Urgency window (time units)', fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(title, fontsize=10)
            ax.legend(fontsize=9); ax.grid(alpha=0.25, linestyle='--')
            ax.invert_xaxis()   # tighter window on the right = more urgent
            ax.set_ylim(bottom=0)

        plt.tight_layout()
        chart_path = Path(OUTPUT_FOLDER) / 'emergency_scenario.png'
        plt.savefig(str(chart_path), dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\nChart saved → {chart_path}")
        print()
        print("  X-axis is reversed: left = loose window (60), right = tight (15)")
        print("  A rising curve (left→right) means drones become MORE valuable")
        print("  as urgency increases")
